# Required31B production-first comparison80
Six exposed development cases, v25/v27,12 counterbalanced generations.
Current exact organizer pins and actual CPU qualification retained. Independent CPU grading follows frozen exports.
Single bundled intervention, not an ablation or holdout.65min/offline/private/L4x4. No main upload or phone access.


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import os, sys, json, hashlib, base64, zipfile, io, time, inspect
IS_KAGGLE = Path('/kaggle/working').is_dir()
RUN_DECLARED_EXPERIMENT = True
SESSION_STARTED_MONOTONIC = time.monotonic()
MODE = 'gpu_comparison'
DATA_DIR = Path('/kaggle/input/competitions/gemma-4-developer-agent')
WHEELHOUSE_DIR = Path('/kaggle/input/datasets/metric/gemma-4-developer-agent-wheelhouse')
MODEL_PATH = Path('/kaggle/input/models/google/gemma-4/other/gemma-4-31b-it-qat-w4a16-ct/2')
if not IS_KAGGLE:
    print('Definitions only: local execution performs no official inference, fixture grading or installation')


In [ ]:
"""Export only artifacts produced inside a run, including per-trial checkpoints."""
import base64
import hashlib
import io
import json
import os
from pathlib import Path
import tempfile
import zipfile


def export_run(run_root: Path, *, emit: bool = True) -> dict:
    run_root = run_root.resolve(strict=True)
    artifacts, manifest = [], []
    for artifact in sorted(run_root.rglob('*')):
        resolved = artifact.resolve()
        if artifact.is_symlink() or not resolved.is_relative_to(run_root):
            raise ValueError('Run artifact is a link or escaped its directory')
        name = artifact.relative_to(run_root).as_posix()
        if name == 'evidence_bundle.zip' or name.startswith('evidence_checkpoints/'):
            continue
        if not artifact.is_file():
            continue
        if name == 'artifact_manifest.json':
            raise ValueError('artifact_manifest.json is reserved for the exported manifest')
        # Read once. A live log may grow, but its archived bytes and hash must agree.
        payload = artifact.read_bytes()
        artifacts.append((name, payload))
        manifest.append({'path': name, 'bytes': len(payload),
                         'sha256': hashlib.sha256(payload).hexdigest()})
    if sum(item['bytes'] for item in manifest) > 32 * 1024 * 1024:
        raise ValueError('Use Kaggle output download for this large evidence bundle')
    bundle = io.BytesIO()
    with zipfile.ZipFile(bundle, 'w', zipfile.ZIP_DEFLATED) as archive:
        for name, payload in artifacts:
            archive.writestr(name, payload)
        archive.writestr('artifact_manifest.json', json.dumps(manifest, indent=2) + '\n')
    bundle_bytes = bundle.getvalue()
    digest = hashlib.sha256(bundle_bytes).hexdigest()
    checkpoints = run_root / 'evidence_checkpoints'
    checkpoints.mkdir(exist_ok=True)
    checkpoint = checkpoints / (digest + '.zip')
    if checkpoint.exists():
        if checkpoint.read_bytes() != bundle_bytes:
            raise ValueError('Existing checkpoint differs from its content hash')
    else:
        with checkpoint.open('xb') as stream:
            stream.write(bundle_bytes)
            stream.flush()
            os.fsync(stream.fileno())
    temporary = None
    try:
        with tempfile.NamedTemporaryFile(dir=run_root, prefix='.evidence_bundle.',
                                         suffix='.tmp', delete=False) as stream:
            temporary = Path(stream.name)
            stream.write(bundle_bytes)
            stream.flush()
            os.fsync(stream.fileno())
        os.replace(temporary, run_root / 'evidence_bundle.zip')
    finally:
        if temporary is not None and temporary.exists():
            temporary.unlink()
    report = {'sha256': digest, 'bytes': len(bundle_bytes), 'file_count': len(manifest),
              'checkpoint_path': checkpoint.relative_to(run_root).as_posix(),
              'snapshot_is_atomic_across_all_files': False}
    if emit:
        print('NEURAL_SWE_BUNDLE_SHA256=' + report['sha256'], flush=True)
        print('NEURAL_SWE_BUNDLE_BASE64=' + base64.b64encode(bundle_bytes).decode('ascii'), flush=True)
    return report


In [ ]:
"""Unpack exactly the archive and members declared by the frozen study."""
import base64
import hashlib
import io
from pathlib import Path
import zipfile


def unpack_frozen_agents(run_root, plan, packages):
    for arm, package in packages.items():
        payload = base64.b64decode(package['base64'], validate=True)
        if hashlib.sha256(payload).hexdigest() != package['sha256']:
            raise ValueError('Candidate package differs from frozen bytes')
        expected = plan['arms'][arm]['files']
        destination = Path(run_root) / 'agents' / arm
        destination.mkdir(parents=True, exist_ok=False)
        with zipfile.ZipFile(io.BytesIO(payload)) as archive:
            names = archive.namelist()
            if len(names) != len(expected) or set(names) != set(expected):
                raise ValueError('Unexpected candidate package members')
            for name in names:
                relative = Path(name)
                target = destination / relative
                if relative.is_absolute() or '..' in relative.parts or not target.resolve().is_relative_to(destination.resolve()):
                    raise ValueError('Candidate member leaves the submission directory')
                data = archive.read(name)
                if hashlib.sha256(data).hexdigest() != expected[name]:
                    raise ValueError('Candidate member differs from frozen bytes')
                target.parent.mkdir(parents=True, exist_ok=True)
                target.write_bytes(data)
        (Path(run_root) / (arm + '.zip')).write_bytes(payload)

PLAN = json.loads('{"tasks_sha256": "e4b3fd60f69dbc2b9213e54eeb9636db78aefe92c1d06269d73d9f5f8f3c8ad6", "guide_sha256": "3d6e57a13234cb4e783ba24eaab486459af76e0923ea4c6607cd41cc8961bbbb", "expected_core_packages": {"adk-eval-core": "0.1.0", "adk-submission": "0.2.13", "swegemma": "0.2.10"}, "expected_core_wheels": {"adk-eval-core": {"version": "0.1.0", "sha256": "f22cdc3fecb3a340f2aa0dde99024ecc9a82128fe91fb10d3e252564b188b289"}, "adk-submission": {"version": "0.2.13", "sha256": "8a19afc752aa6cec4e98e25b09aa1fe4b020ad902af8f014d294c85d70247d5d"}, "swegemma": {"version": "0.2.10", "sha256": "13261f9d5581840f7600d0d66aea10d6a3e4b53661409e072e55f253436c1252"}}, "expected_organizer_python_sources": [{"distribution": "adk-eval-core", "version": "0.1.0", "package": "adk_eval_core", "package_relative_path": "adk_eval_core/__init__.py", "bytes": 4122, "sha256": "528e02f35eaef6fa733acc0a026c80cc191d28d4a4192a740f9b7bc8a88c9211"}, {"distribution": "adk-eval-core", "version": "0.1.0", "package": "adk_eval_core", "package_relative_path": "adk_eval_core/budget/__init__.py", "bytes": 254, "sha256": "4a709384174fb5be5aa22c9a1385ba830d82bf03fab1798280b350311e75588e"}, {"distribution": "adk-eval-core", "version": "0.1.0", "package": "adk_eval_core", "package_relative_path": "adk_eval_core/budget/budget.py", "bytes": 15780, "sha256": "16de25ae326f0f8440772672a4bf7198a4ebb5586c7c3f2b31048ca896f04ab6"}, {"distribution": "adk-eval-core", "version": "0.1.0", "package": "adk_eval_core", "package_relative_path": "adk_eval_core/editing/__init__.py", "bytes": 266, "sha256": "6e4f2c12837a366147d582b9ccfc4f6c3fd0d117e311767e292abe8b84642397"}, {"distribution": "adk-eval-core", "version": "0.1.0", "package": "adk_eval_core", "package_relative_path": "adk_eval_core/editing/edit.py", "bytes": 14676, "sha256": "a78a761d5078df92606c63b3adb61991a2878bcbb80204875d241175cb532a6f"}, {"distribution": "adk-eval-core", "version": "0.1.0", "package": "adk_eval_core", "package_relative_path": "adk_eval_core/errors.py", "bytes": 5691, "sha256": "2a8d545695a89c0abbc8984d771df6ae31155998b8e20cb91da02d0a5b900373"}, {"distribution": "adk-eval-core", "version": "0.1.0", "package": "adk_eval_core", "package_relative_path": "adk_eval_core/models/__init__.py", "bytes": 246, "sha256": "1a9e1673840b8be5bcbf93767a1d130c007abf350cfc2c263e389135bba223af"}, {"distribution": "adk-eval-core", "version": "0.1.0", "package": "adk_eval_core", "package_relative_path": "adk_eval_core/models/base.py", "bytes": 9724, "sha256": "bdedef267b1022ffac970c7bc18c2d0a53753ce1f120a3c355b91c9b7b666a0d"}, {"distribution": "adk-eval-core", "version": "0.1.0", "package": "adk_eval_core", "package_relative_path": "adk_eval_core/plugins/__init__.py", "bytes": 227, "sha256": "dd53fad958a5719da6d6ab6869f59daa7c70060a257dedc77551873557b829f9"}, {"distribution": "adk-eval-core", "version": "0.1.0", "package": "adk_eval_core", "package_relative_path": "adk_eval_core/plugins/display_plugin.py", "bytes": 5648, "sha256": "7250793fa30985086c193fc55d5b81f4c4d115b44ad764e546a8cc1b0ac20d69"}, {"distribution": "adk-eval-core", "version": "0.1.0", "package": "adk_eval_core", "package_relative_path": "adk_eval_core/plugins/retry_plugin.py", "bytes": 7084, "sha256": "ba61cf201167aaf970c3131a4f24b464515509f3cacd5358de99c5ec0fb07af0"}, {"distribution": "adk-eval-core", "version": "0.1.0", "package": "adk_eval_core", "package_relative_path": "adk_eval_core/runner/__init__.py", "bytes": 293, "sha256": "2faaa7d8445299656396987f7b55876025b5dfaab793afa0c11027ab08984b01"}, {"distribution": "adk-eval-core", "version": "0.1.0", "package": "adk_eval_core", "package_relative_path": "adk_eval_core/runner/base.py", "bytes": 1198, "sha256": "fa97acbfdf8be95b9b247e525fb4fa682c8b8da156b405f5bc29f684eafb4d5f"}, {"distribution": "adk-eval-core", "version": "0.1.0", "package": "adk_eval_core", "package_relative_path": "adk_eval_core/runner/runner.py", "bytes": 3321, "sha256": "0c057f402eb42f0bdc4e84fd88d064287f3929284d8f345813df8dedeb7b9cc3"}, {"distribution": "adk-eval-core", "version": "0.1.0", "package": "adk_eval_core", "package_relative_path": "adk_eval_core/sandbox/__init__.py", "bytes": 751, "sha256": "b9f13699e2bc09fa57815bb3c6bf07c4aefb9ab5b74e80ec46625f8421b10d5a"}, {"distribution": "adk-eval-core", "version": "0.1.0", "package": "adk_eval_core", "package_relative_path": "adk_eval_core/sandbox/base.py", "bytes": 7694, "sha256": "da53e3bb0dfb7d2490d63195740d94560291c37001c4ebae4d12733cf4856007"}, {"distribution": "adk-eval-core", "version": "0.1.0", "package": "adk_eval_core", "package_relative_path": "adk_eval_core/sandbox/containers.py", "bytes": 21219, "sha256": "c36a7077901aafcd8a16ae22441dfe40d8713e36bbc8534ab2bad1cd953bf769"}, {"distribution": "adk-eval-core", "version": "0.1.0", "package": "adk_eval_core", "package_relative_path": "adk_eval_core/sandbox/docker_sandbox.py", "bytes": 7011, "sha256": "484b8864c2ff876f10b85f95e69e72b245e8ca5f6f30dc1020baec846212c899"}, {"distribution": "adk-eval-core", "version": "0.1.0", "package": "adk_eval_core", "package_relative_path": "adk_eval_core/sandbox/subprocess_sandbox.py", "bytes": 11715, "sha256": "09dc213bd71f4ad33a2a403732271efbc78bd09db2d2999817d55f4e630ac6fc"}, {"distribution": "adk-eval-core", "version": "0.1.0", "package": "adk_eval_core", "package_relative_path": "adk_eval_core/tracing/__init__.py", "bytes": 408, "sha256": "d1b4ed94c3c5fa97541f593d62645cc73fe60587da6d2deb742c48ac00f01096"}, {"distribution": "adk-eval-core", "version": "0.1.0", "package": "adk_eval_core", "package_relative_path": "adk_eval_core/tracing/atif.py", "bytes": 5302, "sha256": "266324e80f6a4fc23aa3d809f677a6e950a7bc971f8e903ce3f55dc2c0cd8f35"}, {"distribution": "adk-eval-core", "version": "0.1.0", "package": "adk_eval_core", "package_relative_path": "adk_eval_core/tracing/atif_converter.py", "bytes": 7430, "sha256": "1d35490d305e1676d09c11a54301488cdecc6aceb03ee3469831d3f8a1cdd992"}, {"distribution": "adk-eval-core", "version": "0.1.0", "package": "adk_eval_core", "package_relative_path": "adk_eval_core/tracing/markdown.py", "bytes": 7160, "sha256": "e8e07b6a71926196e09a18f08f0cc3ff6da5820bff95fcc0534fdeab083fbb6d"}, {"distribution": "adk-eval-core", "version": "0.1.0", "package": "adk_eval_core", "package_relative_path": "adk_eval_core/tracing/trace.py", "bytes": 23186, "sha256": "b426e17c03b420f38ee4e08c038678ff4ba66d07ea893638f4e22a9fba8a4bc5"}, {"distribution": "adk-eval-core", "version": "0.1.0", "package": "adk_eval_core", "package_relative_path": "adk_eval_core/ui/__init__.py", "bytes": 1571, "sha256": "3f39053c2fd360dedca776c611df55b0a4382e910f3681b0f9dbb3f08b0b459d"}, {"distribution": "adk-eval-core", "version": "0.1.0", "package": "adk_eval_core", "package_relative_path": "adk_eval_core/ui/dashboard.py", "bytes": 18158, "sha256": "c56edca53fe06efb29151ef140b25ac99f795b08faf0bf0632abae18ce544eec"}, {"distribution": "adk-eval-core", "version": "0.1.0", "package": "adk_eval_core", "package_relative_path": "adk_eval_core/ui/event_display.py", "bytes": 23833, "sha256": "4efbb0a55b5cd93cd0d8036c2d3f5f65b67072a0f5dc197e2e7b360a791f4a3e"}, {"distribution": "adk-eval-core", "version": "0.1.0", "package": "adk_eval_core", "package_relative_path": "adk_eval_core/ui/formatters/__init__.py", "bytes": 1170, "sha256": "2502098e77ae11a186177c980cb30560e3cd90703026a0c3dda95e191bbce288"}, {"distribution": "adk-eval-core", "version": "0.1.0", "package": "adk_eval_core", "package_relative_path": "adk_eval_core/ui/formatters/command.py", "bytes": 4046, "sha256": "50856b977eb86fd21f312a340e6d782101b934564a3527348e869294aa84d960"}, {"distribution": "adk-eval-core", "version": "0.1.0", "package": "adk_eval_core", "package_relative_path": "adk_eval_core/ui/formatters/file.py", "bytes": 5709, "sha256": "5ba0b3a5e51925431545a0f2f408725d4e2e487b934f821c635fb364b10511bf"}, {"distribution": "adk-eval-core", "version": "0.1.0", "package": "adk_eval_core", "package_relative_path": "adk_eval_core/ui/formatters/generic.py", "bytes": 3922, "sha256": "f458b8993b7158c3a5a73991c0adb3cb35509cd927a8ab3b14df4a515020b3be"}, {"distribution": "adk-eval-core", "version": "0.1.0", "package": "adk_eval_core", "package_relative_path": "adk_eval_core/ui/formatters/models.py", "bytes": 5146, "sha256": "5445d681fa4f4dc9847cda9388af8c623fd7e1e73bce15f8c42de9e34a8f84ea"}, {"distribution": "adk-eval-core", "version": "0.1.0", "package": "adk_eval_core", "package_relative_path": "adk_eval_core/ui/formatters/python.py", "bytes": 2119, "sha256": "3249c42b0b2d73a447913d07076623f3dbd7401b63726d9e3a3a9777ff5906e0"}, {"distribution": "adk-eval-core", "version": "0.1.0", "package": "adk_eval_core", "package_relative_path": "adk_eval_core/ui/formatters/registry.py", "bytes": 6171, "sha256": "b63f7260e8219d36a3c3f4ebd1001c7d729d1f106446b9ca50620f79a2e092c6"}, {"distribution": "adk-eval-core", "version": "0.1.0", "package": "adk_eval_core", "package_relative_path": "adk_eval_core/ui/hud.py", "bytes": 14051, "sha256": "bfc95127fd9594e5e9269071f3d7241875ab33d026316482cb157111a790757c"}, {"distribution": "adk-eval-core", "version": "0.1.0", "package": "adk_eval_core", "package_relative_path": "adk_eval_core/ui/presenter.py", "bytes": 7238, "sha256": "626d70427727c5e0a8ffcf37032323c3516461640694fd9be2968dbabb8a96d4"}, {"distribution": "adk-eval-core", "version": "0.1.0", "package": "adk_eval_core", "package_relative_path": "adk_eval_core/ui/protocol.py", "bytes": 4050, "sha256": "84d85f3e5adbc74feb7d4bedaf96131541cad5ac57012abb86f9cf4b046feb8a"}, {"distribution": "adk-eval-core", "version": "0.1.0", "package": "adk_eval_core", "package_relative_path": "adk_eval_core/ui/summary.py", "bytes": 592, "sha256": "3bba083cf4d57081c50abb1af49307abe586a39ea1ebf2b9891783331085b2fe"}, {"distribution": "adk-eval-core", "version": "0.1.0", "package": "adk_eval_core", "package_relative_path": "adk_eval_core/utils/__init__.py", "bytes": 437, "sha256": "630451edfc5b0032078ed858d971e8520f2ea776a2231aa5b7dc1a9a1958cd78"}, {"distribution": "adk-eval-core", "version": "0.1.0", "package": "adk_eval_core", "package_relative_path": "adk_eval_core/utils/scoring.py", "bytes": 3505, "sha256": "f6c561e79fbfea17be26c526e96d0ed11a6d57edada61ff2cd512f29eceeb69d"}, {"distribution": "adk-eval-core", "version": "0.1.0", "package": "adk_eval_core", "package_relative_path": "adk_eval_core/utils/utils.py", "bytes": 4562, "sha256": "df512c616a387afdb69eb5841ad8aa7915f042f9c85ab17594f03bf4a27f2df3"}, {"distribution": "adk-eval-core", "version": "0.1.0", "package": "adk_eval_core", "package_relative_path": "adk_eval_core/viewer/__init__.py", "bytes": 159, "sha256": "e477b6d54612929ba547af4d829ece61a74882c7d2d92c29f76f2e97c370bcf9"}, {"distribution": "adk-eval-core", "version": "0.1.0", "package": "adk_eval_core", "package_relative_path": "adk_eval_core/viewer/templates.py", "bytes": 9661, "sha256": "d7ec71cf42a84cb1c196ad9e7b4bef95611f6353a7d22182b7b5f147d9581b03"}, {"distribution": "adk-eval-core", "version": "0.1.0", "package": "adk_eval_core", "package_relative_path": "adk_eval_core/viewer/viewer.py", "bytes": 27504, "sha256": "16e4eebf920d0de53587c4008d1a0c4b365527940aef2f278c0cfc5fc3c14fc9"}, {"distribution": "adk-submission", "version": "0.2.13", "package": "adk_submission", "package_relative_path": "adk_submission/__init__.py", "bytes": 4369, "sha256": "4b89347769780899d22628508e90e1066e5824e586bd1da60c4f817dc91f1b70"}, {"distribution": "adk-submission", "version": "0.2.13", "package": "adk_submission", "package_relative_path": "adk_submission/builders/__init__.py", "bytes": 637, "sha256": "ec6273c54cfb08f22777146c1636be4f13d4d00a67a17d9ae7a19e3a8f18f6c1"}, {"distribution": "adk-submission", "version": "0.2.13", "package": "adk_submission", "package_relative_path": "adk_submission/builders/llm.py", "bytes": 3658, "sha256": "97c27324ce60ac6dc466941bcc19f5482f7693af9fc02fb760a40526c84c10b5"}, {"distribution": "adk-submission", "version": "0.2.13", "package": "adk_submission", "package_relative_path": "adk_submission/builders/sub_agents.py", "bytes": 2801, "sha256": "ba6162832b1ca4e023e2848c6391242c18cbe9dadcfd560a6ebef81108c35b5a"}, {"distribution": "adk-submission", "version": "0.2.13", "package": "adk_submission", "package_relative_path": "adk_submission/builders/workflows.py", "bytes": 5565, "sha256": "d8d7ccf9a51d57daec58c7231e2a62621a3ecca269e71ee64c57062cadadf980"}, {"distribution": "adk-submission", "version": "0.2.13", "package": "adk_submission", "package_relative_path": "adk_submission/compiler.py", "bytes": 10492, "sha256": "39cc4884699faf147ca1b326664e888c1861f750fe36525b840b5f812d593eed"}, {"distribution": "adk-submission", "version": "0.2.13", "package": "adk_submission", "package_relative_path": "adk_submission/context.py", "bytes": 7403, "sha256": "3c77519415b1dd2cc97b42bcbe29d32dd85f8188ca7eba5b5fb8636bce92c908"}, {"distribution": "adk-submission", "version": "0.2.13", "package": "adk_submission", "package_relative_path": "adk_submission/discovery.py", "bytes": 31163, "sha256": "ca19fc75fb3baa42e85438b41248a06c35987cbf26ef652159b302752d0f191c"}, {"distribution": "adk-submission", "version": "0.2.13", "package": "adk_submission", "package_relative_path": "adk_submission/errors.py", "bytes": 13999, "sha256": "7e121e7c6581288788500cd1a8deceb003a74ec150d5477c37f794971a5f4818"}, {"distribution": "adk-submission", "version": "0.2.13", "package": "adk_submission", "package_relative_path": "adk_submission/limits.py", "bytes": 18965, "sha256": "afde5056f2df1fd77b53e0495712d03f60b3e246d4393617dd800d970f7482e2"}, {"distribution": "adk-submission", "version": "0.2.13", "package": "adk_submission", "package_relative_path": "adk_submission/paths.py", "bytes": 9379, "sha256": "7c4b7050141bdb05a9e6849be11f6440f2a18082a364623226c4343649b2f59f"}, {"distribution": "adk-submission", "version": "0.2.13", "package": "adk_submission", "package_relative_path": "adk_submission/registry.py", "bytes": 11173, "sha256": "c17f9770c2f3663309dd61807d2ae4e7d75bbd06b7c674847f655d218e82cf7a"}, {"distribution": "adk-submission", "version": "0.2.13", "package": "adk_submission", "package_relative_path": "adk_submission/resolvers/__init__.py", "bytes": 800, "sha256": "b88fe281c2b927a046e6485c7e0966b1808a5c8d0a0849cd85824ffb04ae53be"}, {"distribution": "adk-submission", "version": "0.2.13", "package": "adk_submission", "package_relative_path": "adk_submission/resolvers/callbacks.py", "bytes": 3643, "sha256": "332ce0732aa3eb1b555630c57683f51e59d5caee3f519371d376210e9ae33e29"}, {"distribution": "adk-submission", "version": "0.2.13", "package": "adk_submission", "package_relative_path": "adk_submission/resolvers/generation.py", "bytes": 18037, "sha256": "ba0d58d1dce4395d656416522855cab07d43b7ed0cd61898234dc35e9b92ecc6"}, {"distribution": "adk-submission", "version": "0.2.13", "package": "adk_submission", "package_relative_path": "adk_submission/resolvers/models.py", "bytes": 3179, "sha256": "39070f6ae52ad8880750b51af373da8762850d3a8ecc65c6f11744afe4cd5066"}, {"distribution": "adk-submission", "version": "0.2.13", "package": "adk_submission", "package_relative_path": "adk_submission/resolvers/tools.py", "bytes": 10593, "sha256": "ae44a5ca815a05edfd763b92283d0444c77755afbd827fc35bcb87eb2633259e"}, {"distribution": "adk-submission", "version": "0.2.13", "package": "adk_submission", "package_relative_path": "adk_submission/schema.py", "bytes": 17404, "sha256": "6554de60cfa208dcb0b095f4236271795b2955adf87f3be3c40dfa3d62521c5d"}, {"distribution": "adk-submission", "version": "0.2.13", "package": "adk_submission", "package_relative_path": "adk_submission/server.py", "bytes": 35447, "sha256": "d60b3fe34dd7f6e359b003db82b68434757c74b1030862bacda5cff6164f91eb"}, {"distribution": "adk-submission", "version": "0.2.13", "package": "adk_submission", "package_relative_path": "adk_submission/yaml_loader.py", "bytes": 12618, "sha256": "01d644ef7f5acc216506245fd7e115e7096641e62e825c6873c83e148b23c69c"}, {"distribution": "swegemma", "version": "0.2.10", "package": "swegemma", "package_relative_path": "swegemma/__init__.py", "bytes": 3858, "sha256": "9d65aa619c1df4895d952a8b58aa9fa2dfb0e0d75edf8c5f41c9bc1a0cc06bf1"}, {"distribution": "swegemma", "version": "0.2.10", "package": "swegemma", "package_relative_path": "swegemma/__main__.py", "bytes": 145, "sha256": "5cd5c92ee1a230e24c17d693c3093edf0e3e78b96c08815351ba82432a1aa4f4"}, {"distribution": "swegemma", "version": "0.2.10", "package": "swegemma", "package_relative_path": "swegemma/budget.py", "bytes": 1305, "sha256": "60bd0f73e0fcede68a3024a5e37836b0bcb0d133952044d7dc3e5a4102f2c2f9"}, {"distribution": "swegemma", "version": "0.2.10", "package": "swegemma", "package_relative_path": "swegemma/cli.py", "bytes": 5221, "sha256": "7197c5fd6b141d7df805dd350130944c84b0b1fc14ed5eeef3f52b2e26d5e30d"}, {"distribution": "swegemma", "version": "0.2.10", "package": "swegemma", "package_relative_path": "swegemma/config.py", "bytes": 7440, "sha256": "f035916218f895796c4d39f1f9ec25284152a6df57cf6de1fea4931165d2349f"}, {"distribution": "swegemma", "version": "0.2.10", "package": "swegemma", "package_relative_path": "swegemma/context.py", "bytes": 23710, "sha256": "97f06280c50df3132d1af7ef6126884b66b48887a1cf94b293af01e5907dc2a7"}, {"distribution": "swegemma", "version": "0.2.10", "package": "swegemma", "package_relative_path": "swegemma/deduplication.py", "bytes": 49745, "sha256": "f8353a8da887fc45823b9174f4578cd816b341d52c3a074964c1c2310af1573e"}, {"distribution": "swegemma", "version": "0.2.10", "package": "swegemma", "package_relative_path": "swegemma/difficulty.py", "bytes": 16760, "sha256": "0b97e060fa6a3b437024a45c42cb131526d5a48b74dd858e4fc0259301f7a6ba"}, {"distribution": "swegemma", "version": "0.2.10", "package": "swegemma", "package_relative_path": "swegemma/display.py", "bytes": 3976, "sha256": "d2dfff4ebe0ff655a240b5c1f073e85702869f618c0aa5c95cc6b54b20be47c8"}, {"distribution": "swegemma", "version": "0.2.10", "package": "swegemma", "package_relative_path": "swegemma/edit.py", "bytes": 2573, "sha256": "d34acdff4530ed211325117395cf7fa9d085f7ee84250d02acf9c8533657a956"}, {"distribution": "swegemma", "version": "0.2.10", "package": "swegemma", "package_relative_path": "swegemma/evaluate.py", "bytes": 25197, "sha256": "96a46f49267ffcf861f7f90b98efbe828d03f27553ceefce745819028589de1e"}, {"distribution": "swegemma", "version": "0.2.10", "package": "swegemma", "package_relative_path": "swegemma/graph/__init__.py", "bytes": 1420, "sha256": "992e9fec50fabd65a003769fc4289dad787b22ca7b6045ce95bf61d3b5e35d0d"}, {"distribution": "swegemma", "version": "0.2.10", "package": "swegemma", "package_relative_path": "swegemma/graph/embedding_utils.py", "bytes": 16080, "sha256": "c0970058b7e069bd4493bd86beb68d29047eb0e51d38814a1a146c428e243bf0"}, {"distribution": "swegemma", "version": "0.2.10", "package": "swegemma", "package_relative_path": "swegemma/graph/graph_utils.py", "bytes": 19051, "sha256": "ab2fc0349207baf0c37423f9a28a550923a0ac553c4de2cf0e58447e96ad3050"}, {"distribution": "swegemma", "version": "0.2.10", "package": "swegemma", "package_relative_path": "swegemma/graph/retrieval_utils.py", "bytes": 34709, "sha256": "65fbdddccef339ee74b6434f254e6a94a5d878aab1cc16a432906944e9dcb8e7"}, {"distribution": "swegemma", "version": "0.2.10", "package": "swegemma", "package_relative_path": "swegemma/harness/__init__.py", "bytes": 1079, "sha256": "ca5053d64fc35f7d457974879656e63fa7980a7b36d37a28bbe7391f1fa90fe5"}, {"distribution": "swegemma", "version": "0.2.10", "package": "swegemma", "package_relative_path": "swegemma/harness/agent_runner.py", "bytes": 45475, "sha256": "3a25b4da391dda93418a608574bac21bd4c6e12be0d144a2fcb8ae178674c0b7"}, {"distribution": "swegemma", "version": "0.2.10", "package": "swegemma", "package_relative_path": "swegemma/harness/container_setup.py", "bytes": 40661, "sha256": "3192acdce35f2d0bd0b191186851b46c251390af16b7f139a1640e16fb1d862c"}, {"distribution": "swegemma", "version": "0.2.10", "package": "swegemma", "package_relative_path": "swegemma/harness/sample_verification.py", "bytes": 20399, "sha256": "0bbd4322534442420a4f782ffb76a5467d6a9d9bd7c32fe212269f304e05fa41"}, {"distribution": "swegemma", "version": "0.2.10", "package": "swegemma", "package_relative_path": "swegemma/harness/verification.py", "bytes": 26984, "sha256": "766e52c34da16cfdc73fc6df22848ab5b9d6b80943d61e7173a844e8efc6c1d9"}, {"distribution": "swegemma", "version": "0.2.10", "package": "swegemma", "package_relative_path": "swegemma/models/__init__.py", "bytes": 793, "sha256": "34ecd27654b031cd3ede1cead44ed0e8f72bbfe6ffea24c22bfe66d493edbd97"}, {"distribution": "swegemma", "version": "0.2.10", "package": "swegemma", "package_relative_path": "swegemma/models/discovery.py", "bytes": 5681, "sha256": "70ff2d1719f71cd4c0d747fecfcf5763c1fbbb983684cc2d68b6771e3a3997ca"}, {"distribution": "swegemma", "version": "0.2.10", "package": "swegemma", "package_relative_path": "swegemma/models/registry.py", "bytes": 7374, "sha256": "69e62c69d461a11f1655285652872005aee11aa549beaec331742dc742593ca3"}, {"distribution": "swegemma", "version": "0.2.10", "package": "swegemma", "package_relative_path": "swegemma/models/task.py", "bytes": 7693, "sha256": "0b9d705ce759645bd1ee59bb28662a25a951990c024bfeb1e00a41f08350812a"}, {"distribution": "swegemma", "version": "0.2.10", "package": "swegemma", "package_relative_path": "swegemma/results.py", "bytes": 5200, "sha256": "4705de95a05ae5f01f743cbb97046fb0f47f1407f4cd1db249489b2253a7f72f"}, {"distribution": "swegemma", "version": "0.2.10", "package": "swegemma", "package_relative_path": "swegemma/sandbox/__init__.py", "bytes": 829, "sha256": "ba3725efee8f6d0fba2321851d66d4497ae3cda5863edc0d4a8ef232f1fdb775"}, {"distribution": "swegemma", "version": "0.2.10", "package": "swegemma", "package_relative_path": "swegemma/sandbox/base.py", "bytes": 10113, "sha256": "aab491aabc84105963bead16dab4d3d5745e31727ba248fd5a568d8f4e0c7dc1"}, {"distribution": "swegemma", "version": "0.2.10", "package": "swegemma", "package_relative_path": "swegemma/sandbox/docker.py", "bytes": 34531, "sha256": "cc10c287acf7c9326f622e91b98207bb9cdcf4fadf5d4aa5f5658e854051d09d"}, {"distribution": "swegemma", "version": "0.2.10", "package": "swegemma", "package_relative_path": "swegemma/sandbox/subprocess.py", "bytes": 21429, "sha256": "a26e9265d93f0387dd0a820ea451d1c530884bebca41583738f3ab2cc31df1f2"}, {"distribution": "swegemma", "version": "0.2.10", "package": "swegemma", "package_relative_path": "swegemma/submission.py", "bytes": 3577, "sha256": "d387b402223300d06eb6f4f960faf82e997098050f5317961c64e6b95365974a"}, {"distribution": "swegemma", "version": "0.2.10", "package": "swegemma", "package_relative_path": "swegemma/tools/__init__.py", "bytes": 1632, "sha256": "e19d1e81d90976fda93f346104d7f34f74018747ac81838a363de2e357358c31"}, {"distribution": "swegemma", "version": "0.2.10", "package": "swegemma", "package_relative_path": "swegemma/tools/base.py", "bytes": 3768, "sha256": "535a0e6d121549b41906685d0e440e8709c4b8b434bc31dcc82fa7740ec66281"}, {"distribution": "swegemma", "version": "0.2.10", "package": "swegemma", "package_relative_path": "swegemma/tools/execution.py", "bytes": 6573, "sha256": "ff4df981ae1070334665ce8ab726c231e139ac5cf34bb5ebc4660eada28a7b2d"}, {"distribution": "swegemma", "version": "0.2.10", "package": "swegemma", "package_relative_path": "swegemma/tools/graph.py", "bytes": 16816, "sha256": "74b74d151925f99ffa37fd68513c234e25b74136369c3a3b072519d36af577cd"}, {"distribution": "swegemma", "version": "0.2.10", "package": "swegemma", "package_relative_path": "swegemma/tools/workspace.py", "bytes": 12316, "sha256": "d38ba2f68636c5e33c9f607bd9e5401dbf81d77125b966ef87c799a1615b05c6"}, {"distribution": "swegemma", "version": "0.2.10", "package": "swegemma", "package_relative_path": "swegemma/warnings.py", "bytes": 1084, "sha256": "f24d5cf69977bd88a5bb82639767ba41c12fed3bd8692969cb7c80051fbb1945"}], "expected_agent_runner_module_sha256": "3a25b4da391dda93418a608574bac21bd4c6e12be0d144a2fcb8ae178674c0b7", "expected_verify_task_sha256": "dc89c3595217c5627d73dcccf41b90a45a2743f2f770537e2f5d2da0b3bced16", "created_utc": "2026-10-09T06:23:41.576719+00:00", "scope": "Prospective matched v25/v27 pilot; six repeatedly exposed development cases, not a holdout", "required_model": "gemma-4-31b-it-qat-w4a16-ct", "required_model_revision": 2, "baseline_arm": "tool_paths_v25_bounded_compact_writer_budget3_v1", "candidate_arm": "tool_paths_v27_production_first_workflow_budget3_v1", "tasks": [{"instance_id": "fastapi_14430", "repo": "fastapi/fastapi", "base_commit": "cff2236dacb7aedcaf2d2de0873cfbc9bb2b897a"}, {"instance_id": "fastapi_14953", "repo": "fastapi/fastapi", "base_commit": "d2c17b603d2f986d5f3800d3bf24acba8e1f485b"}, {"instance_id": "fastapi_14482", "repo": "fastapi/fastapi", "base_commit": "71a17b5932e38bcf798fcd05c5bb384e6d4db5bf"}, {"instance_id": "fastapi_14361", "repo": "fastapi/fastapi", "base_commit": "5265c4f5cb4d8fb4fb5015bccf1a6de8f3fc6044"}, {"instance_id": "rich_3882", "repo": "Textualize/rich", "base_commit": "4d6d631a3d2deddf8405522d4b8c976a6d35726c"}, {"instance_id": "rich_3506", "repo": "Textualize/rich", "base_commit": "4101991898ee7a09fe1706daca24af5e1e054862"}], "schedule": [{"arm": "tool_paths_v25_bounded_compact_writer_budget3_v1", "task_id": "fastapi_14430", "trial_id": "paired_r01"}, {"arm": "tool_paths_v27_production_first_workflow_budget3_v1", "task_id": "fastapi_14430", "trial_id": "paired_r01"}, {"arm": "tool_paths_v27_production_first_workflow_budget3_v1", "task_id": "fastapi_14953", "trial_id": "paired_r01"}, {"arm": "tool_paths_v25_bounded_compact_writer_budget3_v1", "task_id": "fastapi_14953", "trial_id": "paired_r01"}, {"arm": "tool_paths_v25_bounded_compact_writer_budget3_v1", "task_id": "fastapi_14482", "trial_id": "paired_r01"}, {"arm": "tool_paths_v27_production_first_workflow_budget3_v1", "task_id": "fastapi_14482", "trial_id": "paired_r01"}, {"arm": "tool_paths_v27_production_first_workflow_budget3_v1", "task_id": "fastapi_14361", "trial_id": "paired_r01"}, {"arm": "tool_paths_v25_bounded_compact_writer_budget3_v1", "task_id": "fastapi_14361", "trial_id": "paired_r01"}, {"arm": "tool_paths_v25_bounded_compact_writer_budget3_v1", "task_id": "rich_3882", "trial_id": "paired_r01"}, {"arm": "tool_paths_v27_production_first_workflow_budget3_v1", "task_id": "rich_3882", "trial_id": "paired_r01"}, {"arm": "tool_paths_v27_production_first_workflow_budget3_v1", "task_id": "rich_3506", "trial_id": "paired_r01"}, {"arm": "tool_paths_v25_bounded_compact_writer_budget3_v1", "task_id": "rich_3506", "trial_id": "paired_r01"}], "selected_problem_only_input_sha256": "711abc3a0494271c6e65c193f69f8a3f31b0e7eed0e2ed8105e6f64995c84453", "arms": {"tool_paths_v25_bounded_compact_writer_budget3_v1": {"files": {"agent.yaml": "83347c80e6f281774d6923e110fb10f6532bc8dc6cae058886abd952ff47e2f9", "configs/planner_sampling.yaml": "4f07c21d9278bb18102dc13497bec81e9f52b932c26cf0d75ce279b73375e5d2", "configs/sampling.yaml": "2a47816e29420420ef12abdefe80787c8759f096dcdc2d6b52307dbad33cb971", "eval_config.yaml": "f6ed0e7f68df70056e6646df2e2cbe6e3af27686ae0991c26f2ff8e192856783", "planner.yaml": "b27631fde874c029a8345c27ea6ac38b7bc180046f0060668e8e8167818eae6c", "prompts/planner.md": "85ea8411a708049eee9a3c9317c8045ede88244a4e29144fc4cfb572b1dad4aa", "prompts/system.md": "5d73a4447d222b7ffbf9dea4318de6fcc036dbd0a701fe8ba448af9d30cef74c", "skills/source-import/SKILL.md": "8060329ada9f11b74ad56dd403165ae5c1f8c50396b115056e0c031235f0efcf", "skills/source-import/scripts/source_import_guard.py": "482b8dee8690cfe91dbbba19758a89376fa362f86d75546c84317b29aa5f8120", "writer.yaml": "dc11cfdd6a3a9dae9636a618ec1ee009d3862fd6598b52a176db4c7404122f7c"}}, "tool_paths_v27_production_first_workflow_budget3_v1": {"files": {"agent.yaml": "3c9fd67861e43d0d463a85506f00c8a515ca8d72a82425b105b244ff40e0c4e7", "configs/sampling.yaml": "2a47816e29420420ef12abdefe80787c8759f096dcdc2d6b52307dbad33cb971", "eval_config.yaml": "f6ed0e7f68df70056e6646df2e2cbe6e3af27686ae0991c26f2ff8e192856783", "prompts/system.md": "67ec5bd807d6dc6b27a96bd24bcde692fe5dd795e1c8262a226e6101335c3262", "skills/repair-workflow/SKILL.md": "97ec6d673d5040d65c65ce5c7d7f9e9d4b297698ac9c584b780d81f8c9434539", "skills/repair-workflow/scripts/workflow.py": "c2ed1809220e65ee04ea84871446b83a203c8da74c89d165734e555afd073eba", "skills/source-import/SKILL.md": "8060329ada9f11b74ad56dd403165ae5c1f8c50396b115056e0c031235f0efcf", "skills/source-import/scripts/source_import_guard.py": "482b8dee8690cfe91dbbba19758a89376fa362f86d75546c84317b29aa5f8120"}}}, "arm_limits": {"tool_paths_v25_bounded_compact_writer_budget3_v1": {"timeout_seconds": 300, "max_tool_calls": 50, "max_time_minutes": 3, "max_turns": 100}, "tool_paths_v27_production_first_workflow_budget3_v1": {"timeout_seconds": 300, "max_tool_calls": 50, "max_time_minutes": 3, "max_turns": 100}}, "limits": {"timeout_seconds": 300, "max_tool_calls": 50, "max_time_minutes": 3, "max_turns": 100}, "notebook_walltime_budget_seconds": 3900, "frozen_candidate_archive_sha256": {"tool_paths_v25_bounded_compact_writer_budget3_v1": "f1b3bad03ccc10cef6bb4a2c60d9b91c3e8b0c523888408392370d8c3a419a92", "tool_paths_v27_production_first_workflow_budget3_v1": "12675d72152b30a2e58ccc3e17c57f7297967168809976619873392c85cd9194"}, "baseline_selection": "v25 latest submitted official exploratory candidate; its score remains unknown. Selected before this pilot, not assumed stronger than the 0.08 incumbent.", "expected_generation_by_arm": {"tool_paths_v25_bounded_compact_writer_budget3_v1": {"planner": {"temperature": 0.1, "top_p": 0.95, "top_k": 40, "max_output_tokens": 1024, "thinking_config": {"include_thoughts": true, "thinking_budget": 256}}, "writer": {"temperature": 0.1, "top_p": 0.95, "top_k": 40, "max_output_tokens": 8192, "thinking_config": {"include_thoughts": true, "thinking_budget": 4096}}}, "tool_paths_v27_production_first_workflow_budget3_v1": {"writer": {"temperature": 0.1, "top_p": 0.95, "top_k": 40, "max_output_tokens": 8192, "thinking_config": {"include_thoughts": true, "thinking_budget": 4096}}}}, "changed_factors": ["Remove no-source planner; direct grounded writer workflow and bounded production-first context/diagnostic helper"], "unchanged_factors": ["31B QAT revision2/L4x4, writer sampling4096/8192, native tools, source-import helper,3min/50tools/100turns/cmd300, cache and compaction"], "causal_scope": "One bundled intervention, components not isolated; selected public pilot cannot estimate hidden leaderboard or general accuracy.", "analysis": {"primary": "Independently verified nonempty production repairs per assigned episode; all six known paired outcomes required.", "secondary": "First source read/edit/submit, nonempty export, invalid tools, protected conflicts, diagnostic uptake, phase/runtime and missingness.", "advancement_gate": {"known_paired_cases": 6, "candidate_only_verified_repairs_at_least": 2, "baseline_only_repair_losses_at_most": 0, "protected_conflicts": 0, "runtime_feasibility_required": true}, "followup": "Only if pilot passes: prospectively qualified broader reserved development panel within fresh quota; no hidden/winning claim.", "stop": "Unrecognized native/backend error or raised infrastructure exception halts. Exact native budget endings retain empty patch and continue. No selective task replacement/refill.", "independent_CPU_grade": "Freeze every returned patch before fresh clean/generated/reference checks. Empty returned patches are graded; missing exports stay unknown."}, "current_fixture_qualification_review_sha256": "275d52a145a468428f7f96e3df24e41f875cafddc2894b49ed6aaf25c9e3bd38", "tasks_unchanged_from_previous_panel": true, "prior_research_reference_exposure": true, "new_reference_or_grader_text_passed_to_model": false, "synthetic_native_controls_are_not_model_repairs": true, "grading_on_GPU": false, "independent_CPU_grading_required": true, "original_organizer_source_modified": false, "native_compiler_observer_returns_original_object": true, "production_phase_deadlines_mechanically_enforced": false, "Stage1_sidecar_loader_executed": false, "hard_GPU_thinking_token_cap_established": false, "main_submission_selected": false, "official_score": null, "phone_accessed": false, "setup_and_model_startup_included_in_notebook_guard": true, "quota_planning_hours": 4.5, "quota_guard_is_not_hard_billing_guarantee": true}')
PLAN_BYTES_BASE64 = '...'
PACKAGES = json.loads('{"tool_paths_v25_bounded_compact_writer_budget3_v1": {"base64": "UEsDBBQAAAAIAAAASF26OBSzkAAAANAAAAAKAAAAYWdlbnQueWFtbGWNwQ2DMAxF75nCC5QBcmOGDhAZMMWSsdM4AWX7QqWeevvvv8NT3CmCUisoyU9KmTMJKwV8kdY0C7pHeNK7Xcgo432HhXwunCubRhghFzvYL0ABdm8EW89WN3J2WE3ETlpg6qBY+SAolM25Wun3RC5D8Dalb9JjAHjAbLryK2WsW4QsqEpl6LjLvz0L15/8AFBLAwQUAAAACAAAAEhdtNr/QmoAAACAAAAAHQAAAGNvbmZpZ3MvcGxhbm5lcl9zYW1wbGluZy55YW1sPcpBCoMwEEbhfU4xJyhRVOhcJlgdk5CaCek/0OPXdtHdB+9BziZ9hXVh8rfBQVtoX97nnwvT5N25voMamiFAi9QX0+DHySHlWnKNYdN65MiOKNftabsEJLWYcJ3oJlf4vw/bo4BpnBf3AVBLAwQUAAAACAAAAEhdmFcxJlQBAADvAgAAFQAAAGNvbmZpZ3Mvc2FtcGxpbmcueWFtbK1Sy24UMRC8z1eUNOcsSYCQXcQ5FxSQEOdRj6d3prXjh9ztsMuJj+AL+RLsACuR85Zk+WC7XI/u8eGS6Hq4GPYy6ysln1YJ8+ZEfsWvHz/xwIEzmcSAz5TJs3HW+uLSCr4GK4EnqFG2qgDK1nbd4DEikbnlSotzrArxKccn9hwMCylG5gDPpCXztLm4uM7YpxZCpd/henODM3p8+ZsYMnuSoAgxTC0kL0HUxOGb2BKLgbCXY3PIPHUW05Aa2fYt/kePx+JWLop/ZTz3cGBO8FENcY9qf6RRVrETPKk+0x12eHONl+jxUbwYnqKjsayUT9CUmSbsY67LFa2a5nPLnafjUOWmYoPFAwfd4f5me9uYPuWZgnznjLnIxOBjU8jvYdFobUNkfLRzEq9v393dd7ZIOFQXw58Z23WABLeWiYeWy7xY/cFy4XpwvjuWaWZrhrZ33W9QSwMEFAAAAAgAAABIXen1ArBMAAAAXwAAABAAAABldmFsX2NvbmZpZy55YW1sSy1LzClNLMnMz7PiUlAoycxNzS8tiS9OTc7PSym2UjA2MAAK5yZWxJfk5+fEJyfm5ABFTeGCQPXxuZl5pSWpIMUw0dKiPCDXEKgXAFBLAwQUAAAACAAAAEhdk6/TBMYAAAA7AQAADAAAAHBsYW5uZXIueWFtbHVPS2pDMQzc+xTuAZwSGrLwrvveoBSj+ikvprLkWjLh3b4OCSGbroQ0o/kwVIy+qA5M562JnVGLOliRLWUC1eg/qL5fd1dlQYp+xVohHMLb/jsUC79g4XKA/TFkc4XV+shWhKN/KZxpLOhbl9pMXxsBM/ZdXdwdSlnYpva0YWF0MqwNSz+4Rd+xQenPsUyEJvPzy8082MEe/9d5KuuT5+3w8EwKtVHhdbdBJbcUBSK5JOvAepq4SWrQp1T0swH+w0Dseif8AVBLAwQUAAAACAAAAEhdm3tnxF4BAABGAgAAEgAAAHByb21wdHMvcGxhbm5lci5tZGWSwWrEMAxE7/4KfUAI7aWHnkthr6WXnooTK7uithUkJbuh9N8rZ7ew0Jsx8szojV9ZCkTQE4t1MAuvpMQ1ZhCcIwmctpnthEoKk3ABPwOpLggDZj738MELnOKKUDkYc1aINf3dGFDVGUfDBMqLjNjDG9oiFeYcqYLhxWCpCQUenx7gzJIURl5RqB6fQ3PDy01gQFcllm4PkekL8wZU5owFq0Xz3FBILX5hB9GgsBrYmV0hjhYo+RRNhPJvFeO/nN2enisCD4qyxiGjozgKauPi0Wqi5tSHw3SnkBj1uu/VZWtQtzJw7iChjkID7uOCGddYzQ86c1UaKJNtfXjhG6/VBWCijDUW1M4HJxSsIzrBvDRvvzTUXWLJpsDibiM3Eo3TRJc+vO9ee4Xx2BTPlDM0rp7NHyTSMUryTN7sXcnx6LW4thNb/BfcSgvh0NZ8Dt/+RRxJ+bwuNbc4P+EXUEsDBBQAAAAIAAAASF3ikZoRvQcAAEwQAAARAAAAcHJvbXB0cy9zeXN0ZW0ubWRlV02PGzcSvQ8w/6EwJxtQS+s4uXjgwziZILO7Toyxk90gWAhUd7XEqJvsJdmaaQT73/dVkd3SJBd7JBWL9fHeq+IjR9+dmNKBycY4Mu0mqg/G7a3b0xB8M9bJekfRj6FmMq6h2nSd/BrHXW/TdjCpPtCTTQfrrq/EUTyYwA1FjlGO7sZmz2lNX2BwpF2w3HbTisbIs1c+2YZdzSv1b+pEOOYdwx3HZHYd02EaPHxHG8kkMpRsz+ISh03oLAcaOuMc/o/jfo9TcjOMR3fiYFvLzfr66vrqzZp+xZ3U2hATJe87ua7YIsotrktjXNMjm0arErg31mm+JR+9+vqKiB4QdbLtpIbwM5qupLSx/eBDImd6pjb4/qLCkuNutF1DPSfTmGSyN0kmjsOAbHB1YEalUT00J4yuNgnfWkeNr8ce1xqJ+pZsksidl5rUvh86TqzeSmkt8nfJh2lNH7RR2UxqnC2qxgauxYI6i7K5/WqJC8mbZqXuJGh+xrEqTv0OZYuoO9xZbRQhwi389mL2dGBHjrlBzek7r8EFHlgy6Cb1Jje9zIR80Ntw2qPdre043uZvchc6PhmHQ9yiG3oCd6kz2/fcWHhXZHKIK2BhjAg350MCogg3phzn4vK/I35AVXd8MCfrgyLkKxSKWx+yUcbJpykdcCGSWAixos6bZhuPtute3ZRS5qbfvF7JWafhvNMQpTxquo11sEN6lT8IOt7/6XAuN2kFhFsHGOiZuMmG22y43Y8mNOthWk6YsI/vf7upqh5Bdnyzopu7b7/8fPfP7cPHTz89ftn+ePfxXr6tKnZxDJz/RpD+qSpByIU3/3mtHh8ZlAKE/upE2a71aWysPRiGKpYi5ctBoNGphYap/jwYXohlk7YHGFS6Kbr+/vmnH9f00NLJdLZRUGzZnWzwTlCy9a6bBOtgA+eUB0GVm70JpKWngHOzlZbPgNzCrrXPYDvd/3L/+OscaX3g+pipdwf+a1U3kdM4EIcAPC7EmgFCrbEdCgdY683QLMuIHmDxbQtR5IUruNM6ZAdANreAunpq/JMT2GSwZ+UFFY8GgrXKsgsQe9fa/Ri0AjlRNZ8JBP1L3A8Sl4SjQouY13S3cD5rj+mEmYDsiSURIOpZEf5Wwl9o0EBTwwUJ8IeiH5QS8qzpozmycjz2yIZMjByUf0IoDU/CiORzv5FzUNYtDqUc8eBHCN6A02TaBJkW21mwSmjSCPWuPzp+WvopRUkBHV4R6nLQ49DGEotQHAd8J7qKiL8VRKo7NpCoM9GHcdfZmoCmgHFiXdqkaShTBw2Ds5PcawKGoCgxqvhSHuDyH8yDwH7wUQdTIacIocG0sbiLzpIa0JrgxMlE/VH6BpsG8W9SP2SHP8eX8gmY1kF6vSnq52PRwzX9y4djHISU8oXOL9SzrnmQkVnE9QlGMj4qiCZAdOLLOS4Ej4ss13kkLC0t+I5F7m3ubxxxQ4zt2HXT7blQWUQx8ubIy+zUMx888kQhzN55qHC9kC7bRsinyOP0TsepHxO9+SbSoEZvab1e0yuBTr4/4QRUpOrsLpggvdkhJYlpOW3b858y9c0JqUiDXq/pHom8+epr/FDoiWYt0DuHuPoz24GixtbYXL5XAgI0AjXAhgWKKGyG7KcHkFcH9ySclPGkc1dkbKP/4tOAwKIAESJ4C4zBN6SyPpL8NWXvuXuztEZZHYob0cQRsvPg4gBkFeldalsomzVCYjSk0nOBWGVURgO2M4fSbcpiFJbCYctQCcqr0teF+nLyAkCiC/IZCiT5KpellwsNa4PNToX8YLpWvspNLNtT3qcywoTJshUhWoxD0SjxjmImP2BRFKXk56HzWQvfoRqavvqTq3qkf14NivqBH7obiAx2LKOj6FyLqWdf7pJnBub5UY8h6AF+ToSaajxb4RoSWtypDLYtIhFiB4OyZcpkzVnUZV4jUa2UjXUT2bxQeL1mljcAaYdGdGwKuPTA6sxQGRM4LV9j9GPRmeD8dzhfJ993mbaCley69g2/6Co8LgtZwiJ+EtC0aAJQSbrIyyA62AajraxNUoRnNNnpbtuNGoUC5BsorYzQi735Qtt7NrI1QzS0iFKKKpSd4LPsEGXwVDPdMjcXKVI9baE1h1k6ZsovDJk3++UZcd4WVjNG8npaNF0332R3trNpyvN/Fg4aOOA1g5bdnbxtCvqkBBSfQKC4TPZ5EpPMJtRaEIHQdqLPgqRZw6oPVP0bn2rMIC47yHuR/XmP0/u3+nMeo7JgaMIlHzVQEJ0FF/KCRrz9G/gEIEBH5dQelGxs21JV6Zmc/fLtD/d33630Y+lSVWEey4aAdEZF3hGNkNEwUwwdkHt7TNLzYLmYejqPaPJjmVWNWCcQm34fQWbtyoVoZATGF4NWXY5OhxSsZWoBUMKtFw9LEy/yvniwlR17L4pm8qtSPRZhz1Kje5AkLeKkFIdmPR0s/Obi6kbCZSETYboYDWUjkTyivNqAg4AVjmMZnQNGu9ytca7ps0aNzxdK83Lw1odF4e38LM6vpYzk2swv0XMQy+5Yd8b28L6Afl7nPmHBsyKtqI8+MCvdlM8ihyfIHyiFsWF7/vJ/11e/iKdJUpctHvNVHpNmL+o8iyneQrCXCkr39F1/nk2X72FZjv4PUEsDBBQAAAAIAAAASF1Kr/xcWgUAADQLAAAdAAAAc2tpbGxzL3NvdXJjZS1pbXBvcnQvU0tJTEwubWRdVk2P2zYUvPNXEHvpxdIe2l4S9LBB0iYFkl2k27RBURi09GS/rkSq/PDa/z7zSNHW5mLYJvk+5s0M2TSNsmaiVzq45DtqeJqdj6qn0HmeIzv7Sn8hz8NZx4OJ+uEcD87q7kDdU9BlN1ZIU8+Rev3s/FOYTUdLQG1srztnsYpYZhzPmm2I+FLizd79R13UbhhGtoRFzRGBgxuNxAs4v3MnTfbI3tmJbGxVg6rVGxqcJx19CpHtXptaW6QQN7rn0Lkj+RKu1Cmd6sG7KVdsupjMWCtQq3p3icdeTxRNb6Jp9Z+BtE92G554HLcFGf3M8aDLPznuCwA3auCRtrORPXl/uC0btmXDdp+M79v5nBMavw/6n5ummVyfRrrZ6JsPHx/uPz9uP919fHezUVgiG5LPS00D+Nxzs2SUJDf/tvrBhIBomk5cEFnwKDE32roIkIBL9LxLMo4MyEbnKp1X4UAYS+emCTW1+hEY4Z8ZGFYwQwbuN46rOUv9KVBZCoJEyavYRvKzJ3xqE/IcrIl8pOtQT9Sl6HyrP5Ppc4Df/7j/pD2FNMYcGc1ElNRTq9SHWHknOwuOoIggLbwxGC2Fg17lbfXd6BH6jBDeC88KZqocDjK/uRLSCCStvkdw/8wYOTKj+UkKdxbENXpPNglLhexml9Pmo8oNqInDFZbNhdGZJ01jXcO2p1P5mhnWFJbLIPKfPc1hI12rnE5aslclgP+ZrysllNgCRmHsznRPJHRamjZHw6PU2X6HnfO8l/B7w3YDuCcMd2nS0jM+0Sj5nDbZ6CUsRrHfo4XBbdSi6YH9VOLVLT0PAz76zLXuYOyeWo3UlkSJvXu2ozN90OgUdZLtmMJGgaA88MIgOpoxGZACCPolRsiaDvIHOE5eIAsIvJobKsYGExEsRP3TzzqQuE54nYOC0/MIQlwIKLLVncl0h66mJTlggy85N95GnkjtUo+wrX7rynSdBd8tGV8wJ2BoIbQiLY+eBDEhypA5jxpimvWAIUC5IPCdBZwhdR2FMKSx8pyLbFdzXY5UzR7FgBn47uhgjgyp1pjZmqSYqlCxzKuRrSUYHdB0DvuNHlE6OkS5enSAQQm+VfmvMajVAC8WnSe+T77gjVkcuC/JL9WgweUCWbpEKEArwRDlKKkBk2Evx5ES3ORQGDx7Pko9e2968j+El45/lxEV2WW8YcZrCx9cBwfqr+RY2wESSyWt/hU5vZm5VxIHSsMZPWev+lE3b3TzN351BjLZArKBT7/cxmmutp3Vs83LQHjtswotucCg7FkCgFZWGr6gs8yjyuEBVZE/UoW7WBywiCmIyQ2ovEKefdUg2kEustUwRQmFTPPigqM5u4R7D440y1GOKh68S/tDJb3wOmRaSMHLtT0y/G1xvrdl7PV+filSqTq+7CVcs6FZ7NTQLveFH3JzZgkHibKoK4++qEpFf5ZLCrSMyRers4M3uJ0SpuovrJK+nWCC6eZ7vg4ZYP11oEKEarbVASGpict+V9S6Ro8LJau1bmSDotM8cse1x6bcivkJpCdTni1ySx3La8hkT2qyZz58fXx//+nh7vH9injqQjwY4EUGFVxjQbylPFCtaJgGk/2gvFZeCAC0uIK7XS1ty10R5CEEwzDzPJ6LkZ6gDmAu6FKP+f2fGPawlbK3C/cWossQ3n159/lrfTJgUh6vhi43XxizR09Bftb3gboHc/I7g6WtpbrFSLParRG/pCMLiSj7dSbLqhGcQw9QzkYJOrlu40eW26Kg0ax9cXlcQfC+PB+zKi/MnVIUC6kvupeGhR5xmwdotpj5Oq60WE1vNDx950ZasEPVuQ31DVBLAwQUAAAACAAAAEhdxnzZG6cOAAC0LwAAMwAAAHNraWxscy9zb3VyY2UtaW1wb3J0L3NjcmlwdHMvc291cmNlX2ltcG9ydF9ndWFyZC5wedVa/3OjNhb/3X+F6v4AbDFNOt3eTTrMTTbrdjN3l2Syaa+9NKMhINvsYqAInLg7+7/fe08SCIyd7LS9m+MHG4T0JL2vn/fEdDp91SRLUYvEZ8VikaW5YLJoqliwsio2Io9yuI1XIn7PxKOIG+jJ7rcsylm0FHnN5Ps0y4LJ5GaVSibrKE+irAAiizQTrJFCsqJK0jyqtkw290AzFhJb01ymiWD1CuZryjJLga6E0ffFYzA5r1kuNqJi6bosqhppwBqifAkDYWaxibImqovKZxGrxEJUAldZRnW88rFvLWTNIpinqtMil8FkOp1OFC0WVcsyqqSYLKpizZKoFnW6Fnqm9tln+Psb7MSMW0VylaX35lH9QUPQ1GlmWt/JIjf3hVRTwLJwoJnhCh5Nl6qlLlc2mY5Tbcu2va3FukTuts+w0MlkkogFW6a1WxVF7bMXsE3pnUwYXJWomyq3iAZVk7u3DvR2fObMzuBX1hWN9PTQO5/FUQnjBC+aumzq8KZqhE/0xi9cBnQNj498pS/hd1EmhaeXlqQyBn2q+ENRvZdlFAtXPILY47QOL4DNeq3pgplmkHTCXFyYafHYZyFzvmwpOChr5GfXI0glT9LK9TQ9vGIglKJgJQvZbb//HfUSsNBD/YP4IXGBNTTWXoC3jyPU00gqAAPDe7WwANQPLedL5nSE7kZmD8RjLfLEJVqlxxaw2xIMBzQrEPkmrYocKbvO1c83by4vrk5v3qA8HS+QsLvahW6oe1LAWOBr6SltACnjxiSMVC1It50V6XdL6JgCBPKi7l61jD7pcSAu8jrNG9E2VkI2WQ3zoW62o2GZldjMyBBJB2dyVTzM6qLMwO4zx7PnVSQCpcZxAV4jDNlRf17aVRAlmlt6iKwTUMkgETjKBb7UVVqiCOB9kW0EalcaK932PKOAmcjJFiTp23E3URWlUrAfwfmIeVUVleucRTlyBTwZbHuxZej5vgfVbQX7LZi/lGw2s3TGNkq17rKAVWlDUd6XZ0UcofPS9rwukiazjARnrUSwaLJsjW7PrZzb09m/o9lv/O6Xhxfu305+CewG74UzIDK+ofljKWL08ehmH1MJwlyyq229KnLjwPJobTYB8iNdUoS12jmBY/SqAQsGE7pr1SyLtiAQ1DG9L/xFS5BV7FgLA0a9h/gCg9WI4F2R5qjM7guas9OPHeV1zVigynmapzXnQbmF7esXwUNar7hsFov0ERYLr7yBEgN/e3qONjxUdK3bqEW4x67/mGrtjGwlqAjgJJXIQOAbwetCeeLd+cZF9laFa7ldQ/R+D9obbcBzDfXNvkgwQVSW6F3MGnrqTz2eVv9zpRHrRg5sQEMIAgEgkwVsDX01SFkLtG8FNN3t0Z02gRj2kHNwcYbpcAs8ToCfbuf7PPMqaErkvKvc4OvLi5t/XZ/fzF/9fDM/u3w9D51jEL56eXH5w9v59Vt4S61DzlydX/GLS35+8Xr+kx4GLa/P356++sec4/2POPrygp+9mZ/9Hbv09gGL0VtIc8BOgFC2Pfv1MbiRffnYl2NArIBxdgT8nJ2yBQhlBToFcRRgmABFxT+kCFAoSxgobUzMlYwgxipN0E6rqEwTw3uRpOBZiKTymRAZnCF48RGw+Agu6moL86/DDteoO65W7kKfAIDB5vb4zvsWZJku0zyE6BPVEKDX/pSTlXA+9WkvE+21ww9TTUc2cSxEIpLpCUGJqaIxPVH//hR4IcHhTU8w5Pco6zdIfDr1vI8T8RiLsmZz+oM3gPUYBvADcxIY8acCFZfX21JMT/DXFV7AObo0zvVbtQDh3Z68PDq6+zjRsUQrmQLB0X0mQmRJ9+grQVGruvXvIym41Ww9e5OySvPanb69/OH6bM7P/3l1eX0Dmvfj/OLm8vrncPoFiiZImnUpdTiDCAUiJImiC2uBYHAjcLsAsV+nFWgXap2e1dHRZNlEVcJJobjjIbvo3nK4gAwRFAww4mCLEKhfUbiW9PsT/oIqElkzI/tC0T6EFelyZjGMR9005oGg8yEJlcWAfYS2jRA07D1L8qGobgTgbLfx1ORj2LZDsC8HCBZHYGZEeJBSJIpleAPODRk2QBqkRzIEjFNm5IJVaCQa7sAfo8eFdqCAgQ0FCy59j1I4Gq8imsNZLUgErvoIXSx6b5rnae/9HXkZ0EJRgfDqNt3qnBdbRKBfSevkNJQjzcyKKJFqKvDcJvaHINJjD50EDVG2DUO0ERFcVY09orfaM0luHCRXmuvcweD7oshcTQqTAoJ46rmDch6iQjN6QBt6REgTtrUhiuaNUlpo+qxrs6xUw3KD1aiDdvD3ACYgfPJoAyxC47CDtn6JPhdgdlPWsAEw/ybNEr4Gz6aMGO/Ap0S4Hg2Dyi3I9R1sIaiLdeYYYZueY2DEZIHQH9PTVsOLfJEugbR+oeXVUqpElPAaMgxXoAJB+Aidpl7M/up4nYLSihHdEDElPWqbgWMA9wPC/vCxh9Xpre6omDBD2OYM0MwIlMhndSWEntIwsBK/NuDTJFtHeRNl8LxJxYMFaTpO2zPTEvUrWKO+M3mRc6KU9MgkfxqNuS0YRZkRMBxj+O7aLwoGhp9BWpdQ2EV1YFqQrez0olWc1VerPLD+fkEhWKSgXBK02TWrtx2eWriKgq5CYbQW31qXNz4P+TRbrTV9v+tlqgkiFxXZjdmDreN69IfSskB0i2g++wA+aQi8b5PHnlK0+S11WWbFveu8CMRyOUvzRaFGl+1Q8gTID2rS6NcFcKAWL3KJDl55kQEIe+Fj+amOskx5+M71f/1S7468MSF7fBWsC5ipyNNYL9n4GQyZh/I15Q0E7EzAqhcLnQerPg62qOz3nipkeP9mfvraabG4NdRy9s/JSHPUndqAQZrJOEXyFyH74MTgBFC8TR07J23pK8iLB9dUv0AZY7SCAtYBieZuZLVqGCd2FclRHIBGw3Vn6NpV/9Zl71CWcVFiL+cUK40zqhdqb9fVJn2dSJVRColgUSEAgpBEJcNlFSUQ1dCHbKIMa4Q7kN/JRY074IAKE4EJEXTbclLpBuUJeAJNIoF1KF0ZEtCaRMLnEdV5aqs7c5QMoeGpnAB0V4tdyejWDL07EB4tXaBBBqfCouoGIEiUoaffckNAjwNrX0DED4czeX2XqDc3nOTWUeQplhqMCf31LLD9PpmR/fRD8uEJeiyGwNxIGJiDORoiFoYxM9sFvV0Ph/FiPIDvYaeK3XpM2BLsNXekwvauFxyRFe2bQUAc2XafeJO3Q7mu1Dstif5+qcUUb8K9fnxQuFuvI1Wu2QX9awL6aYl/WhjKb+UF+OZEPO6m0q19UCcVkZW8QIjtYDA5lUoIuwZ9N8IaI4hxc9MIvvdS7yjU//3t9sKwucaTID1+mJvY2can1sp7K9GhR/+PpB77mQEiAY+Q1hyjQjhICZ5aAA5WaQuXq+irl9+E+ogjUI+uldh4wUo8JulSyJEQsIcwJEFPEIYePcKjdbLBrvZUxbT5EENUig+2QmZkTbY3PbudHX91dHRyt4fdB7yRTpDGMjLKSU1v4xLNpbGbpWo3Sv7zxxIAbzJWbHxiKVqB+vN8zq7FGmIlK/Jsy3LxoFzWrPUJXRYS3Us8lqCCUl2h1zG1pKDvKVDxhVXYNRfhNwD7COH2+Rw2az3TwXoq0vnjSqkIBtoOqvJLR0kduHxusfX7Xb4RP5ryOYVXw9YeCszkjEp5yilSnZr2PthDEEleFjJ93GMnmvZYTcC8UrY8vtWdgxtzqZPJoFpjejaoFvcIKLUwZeVP3ITRbtBQvqs7IFFU4oS0XhLCcPV842QI3FDvJ0HXsPK/a2ehbcXPMkzXYCJdh9yIKl2k2k8M1ngA2z0rgvTcjPIESZoQGpMiA4KGnNopzfrpuYhZtFYlIgCQRB3KK7l0lIdHdZSqWa+VHuIrO79Rrb3ZQH1KxHoS/HaeKN5WeFbgDrMy8Cs6ZfPZ1236ZGPPfSt/AttiKYDrztzQ0BztrZVqzVLLf4cUJsT7sfg+ddm1svaKqDpyeGv9k0bqrLPjdZTmLpbz7ZMHOo2tUJL6E4ngtFo2azCfK3rjJkLGVUoV95DzpIg596yRePzKIz3EdewzT8iwt6UIsXB3cIROGn1T+Eksp79niEr0MaGJaWHAc1SpGsY5Bwdqk5mZuKnXCLmEz4BHEZ4kfE2l4CIFyYYVctU99tk3x9p54YcSFKuIPv3hDJI4O1LuoVpIOPYpBI4K2sdhEgJjxsoZNMj4M1PQoEbVuytrUKPuYXCCKWSMnKRgqBuaBRUMtDpDSq7P4JCXnACrQUKozNoS9mTMeDndOQx0UwcxVLPqDmN0H1MloLe3J3+hIxmSKp2fDErl3/9wev2aziB6pye4AZ8CB38vttI+79fWcdRz/a0pk3P9aoLXdDo9pdAG8bsu6MMlgEq/idz+6onOWb7tPmHZRBmeCiNjLY4RGAusb5IAHYtHbZrdkOcWr+C2eNCdOcbdvVWtg+o01CT97w8zlH5AsJZr7ZC84IePz+jJkRfUvStMgijcnU2Rx9Nrepb3w06Hc2xyyk9n2c/I5D6tRGF/b/BJlUS8dC/tS+zyqv54isiNweCuDEtqjU1ddSJfQdBR63jq86J2EH6Gh6UUNFBrWR6Yn/XxETx11NOF9aCgi0VQnfNbmbX95tZeCCoMTd8tRgfNZ2C+weE7PAw97q2jI0BPVS1gMNSlveNNrIeUQ+OkTs8/xTrQZR0a1O5ZsUYzw64+mbZnFRO7WYYFRePbjG3uWXiLYcDvKnf6ZFVknFL4wZb7CQn94yEyBjpw/ArUVIHa4+mOFoUJ8rzBr00BOyTSVl6yW1T7XRxpA6Thh1Xi+XMwmnNG3+3qMwDzLS2B8ZwV1CfKumBFX83aTvz/GNxR+JipnasTyP8eQPxf4MN9qOFJjNjT710kQV13mv88ZHkAWHa48nmwchRVGlD5B2HK3wMp4a1ZFtUYOEeL51yXGFTt6S2ds88fIVEnfwBT/AdQSwMEFAAAAAgAAABIXRe4h+zaAgAAowUAAAsAAAB3cml0ZXIueWFtbK1Uy27cRhC8E9A/dKCLDYgryRZ0WCABNpGSILblQJsgR6KXbHIHngc93dRaPuUj8oX+EteQhiQfchMvZA97aqqrq+eYfnzOpzomHiTa6p6Dpy///kc3MmX223+u13SbktGm/KZfUuzdgOzfJASmC7qSO/FplPyQEEYxZy5F2k674FTxiQ3PTfevlDx10nrOXE5T6pP36UC2F0p54Og+g9Xvm9ub6+22ub3eXL27XoXuhLJwR6/OXl3W52f12etVAftuT4sanMdH5/peMpBzCjPuNKphe6DN1Rv6Y/v+hrTdS+DVsxdYRQ6ypjh3odGDNHN/qk60zW4sFa/pp4poM1mKKaRJSVNvB85CEgcXRbKLw9JWkGejyHduYBOdK/TyCe9OdqyiJ0DqHA8xIaDdNHwrGu2bhJ4cqifEsaMuc29KwUUX2JNxHsSko5ENgugKcD9PzhvBBkW4xU3vtwsd1xLndu9MWpvAF7tMMlr4Iu1U8p3Uo+dYc2v1Haro7+ssZeXlqqoCKPs1DcV/9UX9+nxXO6s/stWHCz6/rFsrXu54BOKaArvY+JSZ6Jj+jqg7FDlAHu19m2431EOp2qZYtHrx5x5i0OXLChjbezWBAhEtn9rFYj5xhypnaeRT4YziA+cPXTpEQHmpnuSv6QcXWz91qBA7RtNTnUFhw2qeoSiLeaFARr8BqNWwrErTpmggW96YuSdgywLAGF0E73loZ8oIvNRLy0tf8VdWy6gUPykVd4Af8hSVLbPS9651qGPPOYpq8fKvk3+I56KLOlmKmor3x8llyLCTPuUyFCVjVRmO0TU6X1OeYlO0hlWWGCPXzPKUSDpnj9EhwwePIXzUqLFNOoda7hBrZl8tC1Ks06gLDqPfFAc/7CtBE8UN+13K+v0ygIbM4746qvSD82B6tODNwammKbdSuzCmbEdV1Tnlcp00ljkqboHGElhkKLsmNFj+JwNTp98SvgJQSwECFAAUAAAACAAAAEhdujgUs5AAAADQAAAACgAAAAAAAAAAAAAApIEAAAAAYWdlbnQueWFtbFBLAQIUABQAAAAIAAAASF202v9CagAAAIAAAAAdAAAAAAAAAAAAAACkgbgAAABjb25maWdzL3BsYW5uZXJfc2FtcGxpbmcueWFtbFBLAQIUABQAAAAIAAAASF2YVzEmVAEAAO8CAAAVAAAAAAAAAAAAAACkgV0BAABjb25maWdzL3NhbXBsaW5nLnlhbWxQSwECFAAUAAAACAAAAEhd6fUCsEwAAABfAAAAEAAAAAAAAAAAAAAApIHkAgAAZXZhbF9jb25maWcueWFtbFBLAQIUABQAAAAIAAAASF2Tr9MExgAAADsBAAAMAAAAAAAAAAAAAACkgV4DAABwbGFubmVyLnlhbWxQSwECFAAUAAAACAAAAEhdm3tnxF4BAABGAgAAEgAAAAAAAAAAAAAApIFOBAAAcHJvbXB0cy9wbGFubmVyLm1kUEsBAhQAFAAAAAgAAABIXeKRmhG9BwAATBAAABEAAAAAAAAAAAAAAKSB3AUAAHByb21wdHMvc3lzdGVtLm1kUEsBAhQAFAAAAAgAAABIXUqv/FxaBQAANAsAAB0AAAAAAAAAAAAAAKSByA0AAHNraWxscy9zb3VyY2UtaW1wb3J0L1NLSUxMLm1kUEsBAhQAFAAAAAgAAABIXcZ82RunDgAAtC8AADMAAAAAAAAAAAAAAKSBXRMAAHNraWxscy9zb3VyY2UtaW1wb3J0L3NjcmlwdHMvc291cmNlX2ltcG9ydF9ndWFyZC5weVBLAQIUABQAAAAIAAAASF0XuIfs2gIAAKMFAAALAAAAAAAAAAAAAACkgVUiAAB3cml0ZXIueWFtbFBLBQYAAAAACgAKAKICAABYJQAAAAA=", "sha256": "f1b3bad03ccc10cef6bb4a2c60d9b91c3e8b0c523888408392370d8c3a419a92"}, "tool_paths_v27_production_first_workflow_budget3_v1": {"base64": "UEsDBBQAAAAIAAAASV1HiG/74QIAALsFAAAKAAAAYWdlbnQueWFtbK1UTW/cRgy961cw8MUBPGs7MXxYoAW2tduiSZzA26JHgStR2oHnQyE53jin/oj+wv6ScqTEcQ69WReJIw7n8b3HOYIfnvNpjgBHSrp6wBjg37//gRsqjGH71/UabnNW2NTf8HNOgx8t+1eKEeECruieQp6IHxPiROrV5wTbsotexD5tw3PD/SPnAD11ARnraQJDDiEfQPcEmUdM/rOh+m1ze3O93ba315urd9er2J8AE/bw6uzVpTs/c2evV7XYd3s668EH++j9MBBbZc5xrlsmUdseYXP1Bn7fvr8B6fYUcfXsDTYJI60hzSq0cqB21qfpSTr2U+14DT82AJuiOeWYi4DkQQ/IBJRGn4jYp3GR1cCjQsJ7P6KSzB0G+mTvnnYoJCdWqfc4pmwB7Mr4pWmTrxA8OVROAFMPPeOgAtEnHzGAIo+k1MOEaoTIysr9VHxQMBtU4hY3vd8ucHwHyN3eK3VaDK/tUmKT8DjvhPie3BQwOezU3VsXw4NjqisvV00TDXJYw1j95y7c6/Od8+o+orrDBZ5fuk6rl3ucrOIaIvrUhswIcAR/Jus7VjoMvMn7Nt9uYDCmnJZUuTr+sDcy4PJlYzW2D6JkDCSTvHSLxULG3rqcqaFPFbM1H5Hv+nxIVipQ8yR/DS986kLprUPbMamcylzUbNjMM5RoMa8xwKa3FZRmXFap7XJSA1vfNnNPii0LVgxNRcM9D+0M2YJAbpG86mp/abWMSvWTQHWH4bM8sc6WWRkG33nrY4+cSKR6+ZcSHuO56coOU2VT7P2xeDYadjRkrkNRM1aN2jGyNuUdcElt5dqsssQ2cu1MT42o9/otOrD54FtoPmpFUYvModQ7RNvZV8sCVeu04qO30W+rgx/31aBN5Mf9LrN8v2yFRsZp3zRy58NXoMv3qeTCHTkfp8z69If5Dj27Q+a7wS6Xpum9YL1mWmVMYrdDq9nQsTG+BhOe/ifDplG+JPwHUEsDBBQAAAAIAAAASV2YVzEmVAEAAO8CAAAVAAAAY29uZmlncy9zYW1wbGluZy55YW1srVLLbhQxELzPV5Q05yxJgJBdxDkXFJAQ51GPp3emteOH3O2wy4mP4Av5EuwAK5HzlmT5YLtcj+7x4ZLoergY9jLrKyWfVgnz5kR+xa8fP/HAgTOZxIDPlMmzcdb64tIKvgYrgSeoUbaqAMrWdt3gMSKRueVKi3OsCvEpxyf2HAwLKUbmAM+kJfO0ubi4ztinFkKl3+F6c4Mzenz5mxgye5KgCDFMLSQvQdTE4ZvYEouBsJdjc8g8dRbTkBrZ9i3+R4/H4lYuin9lPPdwYE7wUQ1xj2p/pFFWsRM8qT7THXZ4c42X6PFRvBieoqOxrJRP0JSZJuxjrssVrZrmc8udp+NQ5aZig8UDB93h/mZ725g+5ZmCfOeMucjE4GNTyO9h0WhtQ2R8tHMSr2/f3d13tkg4VBfDnxnbdYAEt5aJh5bLvFj9wXLhenC+O5ZpZmuGtnfdb1BLAwQUAAAACAAAAEld6fUCsEwAAABfAAAAEAAAAGV2YWxfY29uZmlnLnlhbWxLLUvMKU0syczPs+JSUCjJzE3NLy2JL05Nzs9LKbZSMDYwAArnJlbEl+Tn58QnJ+bkAEVN4YJA9fG5mXmlJakgxTDR0qI8INcQqBcAUEsDBBQAAAAIAAAASV085LSLDgUAAJMJAAARAAAAcHJvbXB0cy9zeXN0ZW0ubWRdVk1vGzcQve+vGOgsKW2BoAeflEaFBSS2ITsNgqIQqOWsljCX3PBDsv5935Ar2c5FkPgxnHnz3httOXp7ZEo9k4kx49OVH6pNWVnyXWeNYwo8+miSD2c6mdSTosE4M+DEGLzObTLe0ahS2y+bHz5T7FVAEIQKzAuczYkpcoxybp/1gdOSHpMKqcbD711MKuW4pI2LI7fpkkL0ObTcKKdpzHtrWtpzr47GhxvSnpxPyLneCNxxYNcydeaFI/lAvdGaHR2VNVqVLAeVOBhll03zxSstpSkTFicfnjvrT0v6FlFvdrv4bKzdxTaYccqyrjg1MM1+uTabN52xvAMGPc3qpfjhGnQ8z+akwiHSvzNtYuuPHLAyWyx+Zg6yOds8Pn5b0+phQ3err+tHWt19pk/r29U/m/vt7L9ls5H6Ug4uvkeGgvcpziko98z6ssgvLQdJoVexBxQCH7+MgM8k2vvsdFw2W0b90u3BR4lu+ahcIs0dmlvAkmutspZDrBA4gHgUPii9k4KXtJXTTN5xY71/ziOZjhy3aLYK5xuKyY8CMuOmO9A+COiRVWj72qJT7y0vJFgJi7waMCNVUmqOJqCsS89LRrIhHTcd2ND2yh3wO4+jD0mOniudh9HywC6Vti+be+GFbFzrfH+iRH59BvQFSKg7ea/nYM1zvR0HgSOmBiixi2aPtN9ogDXw/bT++367pta7mIJsoG5FFv0vSnqjmOAPQQ3LRjg3QSsRCrRzec8VTGiv2ufyfK1WWHuQqujOI9W9SQFY18dNxObPLKg1JwlQsp5YcTS66EN7YC/SmWCT9i1pZQbqULzc6AxKrxGl8bCFPz/GmwbfojxxyNATIs1LFIXCHW62yGzksBjBOumd0uIeaOg2O3kCviCPwUvSooR+xTtGDtc2KBAIZZ0B4QBFmb2xJp1fDy2bJylK5AhEGNh4PFvbmKPQvUqt7M3mkNkAzC3j6+qvp2+rL7vN14f77dNOpAbp4kDrtWw//Hi6vb+j75unW6pH6W79/cvmbv1Y4kRGV3XE999/gyoJqpQc0a5YIZs38I4qFp+Bb8/I8Wf2hQRAEepH+UpoA2oFdQCEJeWiOVnUxs2pCxx7UDmxJNZMRydwin4F6uBFZvOyLBYdEroKe9JTvZCXOeDRLvgBXVXpcgV2MpS+cwg+VHtIZmCkLPQZFC5lB5OCxFjf4M3KINgVnFTW3nTMFCrBoVGfhtDx4dLkqwR+a1Drc3XqKLuTCkEv31V2ToMh8CmYAlSngKhu3ollMlVg3sGwoE9t1MFhLBVCIgReha8XOAvZLjB13sKDF3kEEUVoddCAe51E4ZfE5VghCx4XS0NpkOIfH2PRgzZddxkyF4rGvB8g1GnmrV9MLIknlIWGgCSdfJ2XhQ/7bKwui+aQw6vdwIdsVugtdYVB4jqi94V39rxsIG7tT87CMcXgrzN4cYKMJTPMMnlV9iKn4qZgI/ICSk8sHRZbKIVFKfvcSIPlchlNcVQtFzzVBcx0ddQ4tXx+Ha+VZQNmF0l+ZOTFk2uyM+6KYbkqEB4Nn+gAjRf0IDDJAngEWBlOllWJmN1lqWJQ29JMbvgOZnoIHoKAyX8AGWSpAFX/eODq9Hcg/PqnRiZdYbtxIlbL0Fnldlt70YMuMQHyC09bq8yrYoRPgTDdRpru41IzsJJZPBG9/m94my4UMjmpw7DGGLEyuHFzMnboOBjuwPU6zhFHmv4/UEsDBBQAAAAIAAAASV2aRYnomQMAAGwGAAAfAAAAc2tpbGxzL3JlcGFpci13b3JrZmxvdy9TS0lMTC5tZF2US2/rNhCF9/oVg6xtpy3QTbJyYuPWQF6I427uLQyaHFmEKVIlKT/+fc9QchJ0Y9DiY858c2am02nlVct3FLlTNk5PIR5qF06V4aSj7bIN/o4eQu8NG0qhj5pJB5/5nEl5fGpCzOPGNOwSxyMOGqv2PqRsdaI6RGIV3YW6GEyv5ckx3KyaQkC1SUyx99t0sM5th7h0srmh4Yso/L/AEry2jredkoPlUrq9bs+6y6z6aJhU3EOCZWfIJlJUO5XJ2ZQp1JRytH6fJkMwpXOvHHk+Oes5kfXJGqa3S26geDw7q6qFTTocOd7Rzxszrm8mN9Ppvz3HC1Zvm4en1SOt1uvNkuZvK3qZPy/XNH9Z0MPyr/nfq9f3m3+Kut0I1vHZasR+eP7jT4rKHziCFtccE11D4Ng3gJ3SB7XndE/z9QfxWXNE/pX12vUQLVBuG5WawimDrlZyb0arTCYgOx8yHTxA5kYqGiPrDFDnGb0EalXWDQ6FKC+GtnOcGZCVT9T2gCe3dwxEGXEjNg2pJPpAVcAORlHwg9cMZo8N64MA07IotFrk4hjL+ePHZv60XT2/vb5/bIVV2dfByK5tO3HY8+ti87T85VWCxTItluvV+3KxvfL85cudxPCmSVj//hsQP+IJEvnwSYJeWJHBinVfGAoakJFKY42NzlltP91cjMX+aGPwLfvBkpV2IUkrZGOHF+CpyACtFXKbkZR17IP4RRqMXFCmwBY7GxrSn1WoB9CHaFLZHD4XZ4/1q4G4bI3+BGTNCYWXdxlRLjQwgo3hfo6TCs3JMC9SlN474+KYNeLuLnlUORpbOENCVvA7JWS0C2ck06mddTZbJIBv1WfV2XwSmlDmhKYD89ru+1gcNkHSJ1+SNSorUZA6dUKcxjpzVc9opLlzFPrc9fk225axvAUJluYvfYHApVxDVe6p9+JXL30sQsBdQaHf0+AJcXcF7zbsOhBDTXqXS9MDQ13jDcCzaLeO8YNy7qNC5tCnvox7nUvVgmsl96/dd0FJRFLwTD9AQFAjL5i8T6JhKGy2x6vBhPyXd2ZlyE2niDOOtm8VBfw+etSmNAc4IArGpa0xixjj4R5IS8r7HuiowWAtDQ6GL0PIyMpsi2ekPGWFb60Six5BSe1gNfrBnlEkLkm0rDyE172Tt3QztkeVBE/JCDXBSXIBf8Dl2+yJAexaWBAjqkdeOaAWu4gRhseVTJGSo2TCR8zQMgP+A1BLAwQUAAAACAAAAEldCczKLi0UAABCOwAAKgAAAHNraWxscy9yZXBhaXItd29ya2Zsb3cvc2NyaXB0cy93b3JrZmxvdy5wec07a3PcyHHfVaX/MKFdBUDCQqQsKpeloCu9Lr6KIykn2rGzx0JhgVnumFgAhwFI7in67+nHDDDAgpQcx1WhfQIwj56efk13T+/R0dHrqitzmYtG1pVWbdXsRVaVrbxtRVrmIlca2tN1IUOxUaVq5UKVdddCR3pZVrpVmZC3MutaVZXRwwcPH/xQNTBTVM1lWqpfZSM0wFlXt0JfqaI4E6W8hsZOw5pVKVJRb6tSRuJTC8PSJheFWjcpYFGVxR4AvtnK7Aqwy6SqWy2qtZbNtRRlugMAaldXTQsvuyrvCgmLqktV6lCUFaBfFAY1GLDet1IDuKOjo4cPeJpIm8s6bbQcGnT78MGmqXZAgqKQGe5Jm0XEGyBUK5t+8DbVW8C1//6rrsr+Y5e22/6j0gZqDa0wxUL86A5qBjR0t66bKpNaD0374b2Vu3qjimF8q3YSKf/p3378KGLx2YsuVeuFwouuZXmNL/aZJPU+S7OtTBL8LKtcJkw6jd/rThU5vgDXGUBb3XpfHj44f/fnc4Zc76m93ivTvyvoJdtc0hNEhJ67nPtvW5z/6fwDY9ZuJbYDq/GxqRp83Kh2i892qzQ/U1odaYbPbXpNs262suR+fD58IMyflzbcn2reVcuboUcuN/jIilRTL9MM3xrZdg0B1LLY8MxSjgC3TUeQN2mh6UW3hLEqCUK7r1VJ+86AktShtZkBDMKd4/8ABaG3qZ+nbRosGTwvboUogu6np895RLSVt7m6lLr1gwFAW13JUvuomBbGTdXkGsjayAhUMwd59xtv9WrxX+ni1+S3F/btePEvv714BEjRXJ5agQbHYnXBX8AHAiZUyUCXAw2K6gbUNab2iD78YOhVGzMAFQ5mE6PRbhSy9KknEC/FiQPPrB6ldS3L3IwZuhEVUEoCBvvSdaFa2FXy+L/971/EK9jPReB/H+PeLgLYE6IVTMDT/JgehxgbrGnMP8UGe8SYWua2gR1zu5jshIaNmAudA//QqKaqlLnfVFUbki2YCAM2RY3UVXEt/SBSOmlkkbbqWiZtRdOGXkcykPcJ0OFK12kmfXkLNMtUG78HabYrgNl7q3RWoekF9QHziWDZJBsjCbqoxb+qVmRocQH3ECiRFV0OIu6YJDS1MiIzioAzoJICsZUoiCu79gWS2H4IpYmuiI+QoEhihZYvym5yPwjJCvrek34Dnm2zdi66lC2+5woYGURAZ1m24gnoez/FyPFkUVzQ4ZmD6uOYcfDrgCUOmV7pCMykauAcgxV97+Nfzn//4f3HV+e/R4X2AiOMMAwZpSXMRUEyayN7kAZatlbcEHK/KK4wYLAcqRBSp+9DvtNWJ+KGAqTKTg6tqJYDY6KmK/2VZ0z/4g38C9bK7+EGZPKuF3Tk0ZCF3lY3i7aqCziSC+8iPJRvd/20BimVCUgGnP/xOZjGkI4eaIh/F7LYxD+gpRwbiCZi+UYbKeJYHE82RpSL0jz3iSVNpNsc9SqXOAMUAbahamS9K/0GOCooAQhQm10VbVIFovantOjku6apGt97k5ak3znIj9qghyFJ4Hs5OkO3ZBChxQIhe2OlZmzrqnaNs666JpNJke4BccKnV20cHhr/hHTkIrxXRox5wH9RxnWTecG3SItwAWGr3u8KVV59gxxt0AV0jgOLF4o5oqTJx/Id6OAHsTYuZwwrqLa7PCJGKN9jAO/HbwKZt4vW2ScMn6Bfg65pkqB7QoYT7YYZhGvfPfAuKrlroq/ZnwzoY/FxMenxPdDrlh0Yye5TXmXaPhPkJLzL23RXo781t2LPjeFQMasEZ1aGSFOG9pFo0Nx59WJ4Aw+nR0sQpTrBEOC2Vy8j8bnKWt9IOAGLjVCzK2UdyNiIifkMHGuCiJKCJfDPVQquTQJEA0byiSRzY03S3VpddlWnY6SpAeTAsRP2yU622yqPvXe34KniAbWB3TwBEguzgnA4zeEA+rSNWlOgsiDGXXZgNtGBC8bHNLhJRkZ/6WCxUDwCBU5vE4oh4u/EI3Fy/PSZeXAXypuOT47hb2JFsVeDJStzHT+Ljnk4GLKuAEI+589sCzY5/g4mu2YDPbvRmd8bPdwOdc0cFYdm76chsiOwO9Xv2oF3LF6QMaUtB+JFLBCfewH/Bw4Vu05jMATexEkU4RyB20kzsBHargHvOzR+BfDAR3mK0Le/kntwaNmv5WWDYLV8/uxihBjNvReP9xVY1LSBZSFMZZ5ZI68cJHTLfiGeWdGuAthVqTIfNAv1k+zfmdCgIWCj4AtZmmQY9OExe6VAgbD52CAHxxnqMRjtWNChdwb6kkIMCN/oePAwPhEQwOEJwSPg9M47ijSTupEbdUunRNsBbH8nHgvviUfGeIc2hqevvLHmeRcBDOQpeGDAjMfi2+ce4AEiRZ4M6/NncxQxdqvl4uSCTwf6RsgzWxhZJf8QQjA25GSj7xjVW+gvBlNU07pREOu36lcJrm9ajO33b8RPMs3FYF4cDK190GItYReSVYLUFw+qIm3AdpA8tI2kjMEA9RwcZzAz4ELfbKtCLpyUSQ3RJdpLOOAADnjYjZDXKIOZFBjeohDkHfoVaTuCipRcp1pO6EgseCxWiN3FxKCbEAnkOxQ4LjToswtL1ECI0FwVEN4g8bTxy2aOHISwWl4M/M5pAYoFCQtgYO6efvOnVs9FdqsJvQC4mY95+JXJRImY7R61TqHNkCmYAbpXssjniMRDrxWwjbR54ondQdZ7pQ3n0XEyeEo0ccY3mloesTBG6WXsnhMoiI71MZ0EdM5LcS0RnqOOIfJwxaRqaHKyRu/AOxNr6L+ahszgH8VTcuO25t0w3W1Q+Y1YUHrIOIz/AN8PXBUK5yk6vttrOcAU5wF2htvfvJwZT34WgJjAdRgDUeTJuFODiFhUgbMtBS8Jth4gR0Nfiqenz40jMYOfPXVwobO7HWN7Zj22QHtn5W8VF5x0n5i0zX4GZJPeDPxJc14az1XKIsfYb4M5r2s3i++8CU3lbSZrUP8Pn+hED8UfQT1gOH3Nycm3EKanSsxBYnozWZWYiAedyez6bootOPvfiR0eHdbZRunhrFtoFoPQr9v5/B5dowsDdILAHLODiGAwducHXk1o4LAm37g7AF+JDFIS4v/tovhF9hzPNeymcdDAfpnamJEr/LYOQXp9SfmFnZ+vfnfhHAoABI0DCJl/EhJtqQkplpZX0o0kcdIMEWDOZbtFM4tZTgvU2SMc3A3K53F0PDa0I8QnRGmREu5GDu3vZk4hiIKYsI+K6tI/ATXy+02Bhc43DA3ao1PcuD9umTl+GH0QPIT9CBF7JJ5GT3EuvD8GR/kptPjR01ME8c+n8MEkGcgK1A+moHEHZv9AVJvanFMPu/5Tl35kKaBjei6BJA/uUSzWVVX4c14pnse4Lh1ZGr2auWFTpH8jPkKPbFwfzHpEoPcmv1h3a/CNBEXQT1hM8cLE+lDRGCbLWa9mvF0g6El0Smoz2hFlHE+iY1AzRxRZ9MLx2D7wZfh4mvsQpMQFBKZ5KvKl8CmSXD27CMUiXx3DI1+Bn9oHzBTWmVCCrrj6gAGllxD9OhrEYEJhtXSiRdcPhOOV4wSAwnlJakGju1WMgUJBo2VVSOMRrCy7nWzAB/VpPGUw03LvkxJik5Uq1rZe1YKL8dqwAK4DFEAItCTTGSKo/W5dFSYKOhNFFUI/aRjbC5y9EKfAjp0qSdMYFdMFSD9zRIihJZISVFs1TiyCMEIHiyJdTU2VYf68wmsvIlHJzhvuMtUtu874QilScw6wQIFb36YgiH4ZChryQ1cSr97KTUiTX+l9mR20vsFrJ/gKTJwT4R7LCsNr2i08S9xDwu0XM+Zpw/jekS0qKalKpKRhoXAEtlyOoAPZLQJ3OOI9sxCWGRr2wB1QoTsCeHZ6Gjicp75D53GWpxPA847Bpz34j7fGORhC/xACvaxrNFD8TlcBI7OhFQE2NTol3s+lF/21gq1tvM/qy1J8JjFcKYg7v3DYrIwiXoK2MGFAOF371sgdeLVov+IhjwNURs2fSKod+OIw/z3xs7LCphpQNwzCeP/UA5nbjnlbLftR1gI9HgNysSezYs0opfnQ24nJPpmEBV9MxtZOMetifoQcvRDnYqDQvdcHFsPYPEPbkLQNaA2YpDy2WwfaA7FfOILyFdiD7YxHljQkOwrESMj0xg26tr4xw8+C2UQnuO2djr3qygtNziTmh0kHJmQQY/o3tESMzdPBk+I1COlkGQ8xgwkqE1Xm8hZ23DsaIVcoJMZrjc3TgWd8XpNqNF8hubnJ4CjGw6vp021VJ+zix/wIpzIAWLpsYHcZzx8UO/ri3QGW4AjXIFV9SvOukNZFnCUJM74yx8jKZHsJu6rJZRN7b78tV9OrkZN6sT6Ce3FvuB57Hwdg4OFlW4C1gdge+LA2dS+FvFVZWojX/07eGCrAmXj16bzXLaxDGcGWhbxOyzYxqWLaGEhUI7MWDIim3TkZZSBrQndjJvbFwMZeCFFq2ZLy5Ni5rf2pK50SG927THorC4xqQfzqCoOSqjwTeaNMTY12CnaAEVUD5KOqHAT7qavBU4A9f9wDsHLIVzXgg2HJja3UydI6XStwKpTUkTjfKm3vcFPEtmtUuzc5VTOj4mvlarNRmQJqXjZpbu7VKT9s8vl46wb8bYFUgAZ7e7D5jbq0ObH+Yvn+DDgKJRISBfT5M0w6m0wDprFtGgVs7dO+A4skuqIgKXDKJC5+vnnkf7/8OXIbAqyWYA7dn1r/sYQgTuXE0yc84QlqBEvXkP4euSGon3g9xYLgvQAWle1iYPZLviaSmZ1uTkHnEATfAltd3GZMmGLskprY7bGDhfa3P9mBX6D61MSvd9jZHcg1oEkDd/oy7OuqOK8YChA/TLbnYGjqVDWja+Cv58CBoUw8c8XuRV6ArqVJL92Vuf7K9rtSg8BTRs6siZkwnu0dbmFi6+/bEaeCjdaZijSIjwrMfuynRWnQscGccV8ldyZUy1VwFhrP0KxCDA60A1PRrTV92KWt/g6yYrTGTMKCAM/zhpqxEIxPvQ+xNC00pUZh26SZXINlva8oDWUzwZ7QVN7xB+8Ijn/wGprL69XJEjxWMKBoAHMZQ6gziDotjUUJNNV3IEITOofxUZKgPU+SowChGCHXrdy9A5BjGR8WQRmkUoKxY26bQ4W5SwpG/GOuCLHjtVOCcjIs+TrV8h29AnOWVmMNkaK6AYAJDIXYakDiBIgHNrMo7G7hXNsRXewtLZaQGBNyxmWJMTSlbdv4ODaEzZNbkByFVKhjbmFiSsHyBKfgATfCjYz+cMNk+BN/PuLVjpbmaDni8UdLrAIxtSrD5PCIPbuEhJMPqKPlpBytHuXzRoVpDI4uwp37dgc8H1Ggl7IBFIAwLP9USWpQTYxCW5E/WlLuAckTfOHdcUWII4JBdAOnj0zokhblOsq7Xa3tmCCUWGeCZ/8RpRpRtHh1sCs9/6AR9OT+a0HuxXO3r7iMziViDEflW4XHPbgfPuc9Yo/NRDKoZuIFKMI6a/DIGSW3qMWm283nOPM3TZjbQdO7/YMT6e1QlmvXoQN4LbEcTYOr5PhObh520E9j2EIrWmi5DagnwiNJ4TJQp5WnHDSzCEbIJ29mKZeXfBr23LOJYuvcuEOttZsZ7oRG5TUgTgfCUNg17l+5JV54AzaUdnFkiJpjbyWdQjF7Hrk1Ef096gT22w/vz//zpx/P373+y/m7Nx/evqOFvBOHHFzuZGgPH7JpEnMf45DSFEVhQW04aYcJ1D6AJKl14EYVhHi+d7M2Mkk9o8UOR0DPtK7SVACOqs4+0sTVRMOFt3hNNWZ/xn9t0bHVFbypANK6UoX9XjDjfuBAFoHAFLRZ6THfveab8/YCXOybPGa9AU7E8N8sXBCc2NnI23d/ev/HP/whNNSJx0SK+THJm8zc+Nu/+dTTzRbMiCVkBB58gab0oFTR/fvKvaFxdu+YPELSM9V683c7znqu5IyvsYhzg9BMOl+6hTHfhBEXE9535dTTE2mgCylrPzp+enpwK+cexpPt/E3ktoOxMnbuvtr236SqnVRC87amJvrQMzVR+OCBcjw8eBd0ghoPNKMYD73Zq7K6GdW9T/++OTa/17nFP+Q+1qKjqjuy4HoDqyWEXSdOwhKFYpjTi8i9c/5Bsjabz3El7f8z7UmQyKQNjtX0yqa8hpARkx8x/ewkKqo0732gsdM2C2/qRPZ8ZxfTeCb9MitzkGOhEV1L2uXR/5sZdRf4XzqIR3Hv9prJrIdgph6PcYEPDO5sps7V8kmxMePCbkuySWH7h+w/ZOsgD4dg+xPCasnkcjtEVagaTT5hQRXtJPjHcwXW5nSxyvP3gDISPuTx7lGtl8ImTe5VsWeH9Yz4x8Q0lElMdNPLQcg/zjKRdmK8pJ73cf82A5kSFqR1iXHZdgA0wRyc0VSbYko4xZTIXzoFszADaCL5GW7+fco5x7YM3J7Y+2CDddrxwvwKDXXMBP1DOHDW/ybNRvsV5gTSrO2FcYE6ayoX25FzPif2poyTQ7fEqPh9lm1Gkuft2R3G7//AxtmcKIb9vVWrwRrYn+JFr5rLDln+Eb/A85PgJCqKzOMkyassSWykjPU5SWqG+x7woyGMPfwpQqVgq/HKszWA9BstJNRQ7ziZb4r9Q9HugbNo/+4cSNcA3p3d9FuwO3tN/unOfkNci8gGTDveE8tN2hUt5ohNdQaWpvBFJU7X1hUxadPJb4LSaJxtS6OeXGgkBzKNUmqY86eSCrckOo24wBak1+sTlMDduZnT1HfKiRiail/9rzbp24pVGpm3vjJVlZNYH+Gjc6/xBympzpTirPv4PueYzz4cDMEaqQ1GX+h2wpHBaSCWSRiXUE4qSajfJqYsPTjOHnJTPgswLPc/UEsDBBQAAAAIAAAASV1Kr/xcWgUAADQLAAAdAAAAc2tpbGxzL3NvdXJjZS1pbXBvcnQvU0tJTEwubWRdVk2P2zYUvPNXEHvpxdIe2l4S9LBB0iYFkl2k27RBURi09GS/rkSq/PDa/z7zSNHW5mLYJvk+5s0M2TSNsmaiVzq45DtqeJqdj6qn0HmeIzv7Sn8hz8NZx4OJ+uEcD87q7kDdU9BlN1ZIU8+Rev3s/FOYTUdLQG1srztnsYpYZhzPmm2I+FLizd79R13UbhhGtoRFzRGBgxuNxAs4v3MnTfbI3tmJbGxVg6rVGxqcJx19CpHtXptaW6QQN7rn0Lkj+RKu1Cmd6sG7KVdsupjMWCtQq3p3icdeTxRNb6Jp9Z+BtE92G554HLcFGf3M8aDLPznuCwA3auCRtrORPXl/uC0btmXDdp+M79v5nBMavw/6n5ummVyfRrrZ6JsPHx/uPz9uP919fHezUVgiG5LPS00D+Nxzs2SUJDf/tvrBhIBomk5cEFnwKDE32roIkIBL9LxLMo4MyEbnKp1X4UAYS+emCTW1+hEY4Z8ZGFYwQwbuN46rOUv9KVBZCoJEyavYRvKzJ3xqE/IcrIl8pOtQT9Sl6HyrP5Ppc4Df/7j/pD2FNMYcGc1ElNRTq9SHWHknOwuOoIggLbwxGC2Fg17lbfXd6BH6jBDeC88KZqocDjK/uRLSCCStvkdw/8wYOTKj+UkKdxbENXpPNglLhexml9Pmo8oNqInDFZbNhdGZJ01jXcO2p1P5mhnWFJbLIPKfPc1hI12rnE5aslclgP+ZrysllNgCRmHsznRPJHRamjZHw6PU2X6HnfO8l/B7w3YDuCcMd2nS0jM+0Sj5nDbZ6CUsRrHfo4XBbdSi6YH9VOLVLT0PAz76zLXuYOyeWo3UlkSJvXu2ozN90OgUdZLtmMJGgaA88MIgOpoxGZACCPolRsiaDvIHOE5eIAsIvJobKsYGExEsRP3TzzqQuE54nYOC0/MIQlwIKLLVncl0h66mJTlggy85N95GnkjtUo+wrX7rynSdBd8tGV8wJ2BoIbQiLY+eBDEhypA5jxpimvWAIUC5IPCdBZwhdR2FMKSx8pyLbFdzXY5UzR7FgBn47uhgjgyp1pjZmqSYqlCxzKuRrSUYHdB0DvuNHlE6OkS5enSAQQm+VfmvMajVAC8WnSe+T77gjVkcuC/JL9WgweUCWbpEKEArwRDlKKkBk2Evx5ES3ORQGDx7Pko9e2968j+El45/lxEV2WW8YcZrCx9cBwfqr+RY2wESSyWt/hU5vZm5VxIHSsMZPWev+lE3b3TzN351BjLZArKBT7/cxmmutp3Vs83LQHjtswotucCg7FkCgFZWGr6gs8yjyuEBVZE/UoW7WBywiCmIyQ2ovEKefdUg2kEustUwRQmFTPPigqM5u4R7D440y1GOKh68S/tDJb3wOmRaSMHLtT0y/G1xvrdl7PV+filSqTq+7CVcs6FZ7NTQLveFH3JzZgkHibKoK4++qEpFf5ZLCrSMyRers4M3uJ0SpuovrJK+nWCC6eZ7vg4ZYP11oEKEarbVASGpict+V9S6Ro8LJau1bmSDotM8cse1x6bcivkJpCdTni1ySx3La8hkT2qyZz58fXx//+nh7vH9injqQjwY4EUGFVxjQbylPFCtaJgGk/2gvFZeCAC0uIK7XS1ty10R5CEEwzDzPJ6LkZ6gDmAu6FKP+f2fGPawlbK3C/cWossQ3n159/lrfTJgUh6vhi43XxizR09Bftb3gboHc/I7g6WtpbrFSLParRG/pCMLiSj7dSbLqhGcQw9QzkYJOrlu40eW26Kg0ax9cXlcQfC+PB+zKi/MnVIUC6kvupeGhR5xmwdotpj5Oq60WE1vNDx950ZasEPVuQ31DVBLAwQUAAAACAAAAEldxnzZG6cOAAC0LwAAMwAAAHNraWxscy9zb3VyY2UtaW1wb3J0L3NjcmlwdHMvc291cmNlX2ltcG9ydF9ndWFyZC5wedVa/3OjNhb/3X+F6v4AbDFNOt3eTTrMTTbrdjN3l2Syaa+9NKMhINvsYqAInLg7+7/fe08SCIyd7LS9m+MHG4T0JL2vn/fEdDp91SRLUYvEZ8VikaW5YLJoqliwsio2Io9yuI1XIn7PxKOIG+jJ7rcsylm0FHnN5Ps0y4LJ5GaVSibrKE+irAAiizQTrJFCsqJK0jyqtkw290AzFhJb01ymiWD1CuZryjJLga6E0ffFYzA5r1kuNqJi6bosqhppwBqifAkDYWaxibImqovKZxGrxEJUAldZRnW88rFvLWTNIpinqtMil8FkOp1OFC0WVcsyqqSYLKpizZKoFnW6Fnqm9tln+Psb7MSMW0VylaX35lH9QUPQ1GlmWt/JIjf3hVRTwLJwoJnhCh5Nl6qlLlc2mY5Tbcu2va3FukTuts+w0MlkkogFW6a1WxVF7bMXsE3pnUwYXJWomyq3iAZVk7u3DvR2fObMzuBX1hWN9PTQO5/FUQnjBC+aumzq8KZqhE/0xi9cBnQNj498pS/hd1EmhaeXlqQyBn2q+ENRvZdlFAtXPILY47QOL4DNeq3pgplmkHTCXFyYafHYZyFzvmwpOChr5GfXI0glT9LK9TQ9vGIglKJgJQvZbb//HfUSsNBD/YP4IXGBNTTWXoC3jyPU00gqAAPDe7WwANQPLedL5nSE7kZmD8RjLfLEJVqlxxaw2xIMBzQrEPkmrYocKbvO1c83by4vrk5v3qA8HS+QsLvahW6oe1LAWOBr6SltACnjxiSMVC1It50V6XdL6JgCBPKi7l61jD7pcSAu8jrNG9E2VkI2WQ3zoW62o2GZldjMyBBJB2dyVTzM6qLMwO4zx7PnVSQCpcZxAV4jDNlRf17aVRAlmlt6iKwTUMkgETjKBb7UVVqiCOB9kW0EalcaK932PKOAmcjJFiTp23E3URWlUrAfwfmIeVUVleucRTlyBTwZbHuxZej5vgfVbQX7LZi/lGw2s3TGNkq17rKAVWlDUd6XZ0UcofPS9rwukiazjARnrUSwaLJsjW7PrZzb09m/o9lv/O6Xhxfu305+CewG74UzIDK+ofljKWL08ehmH1MJwlyyq229KnLjwPJobTYB8iNdUoS12jmBY/SqAQsGE7pr1SyLtiAQ1DG9L/xFS5BV7FgLA0a9h/gCg9WI4F2R5qjM7guas9OPHeV1zVigynmapzXnQbmF7esXwUNar7hsFov0ERYLr7yBEgN/e3qONjxUdK3bqEW4x67/mGrtjGwlqAjgJJXIQOAbwetCeeLd+cZF9laFa7ldQ/R+D9obbcBzDfXNvkgwQVSW6F3MGnrqTz2eVv9zpRHrRg5sQEMIAgEgkwVsDX01SFkLtG8FNN3t0Z02gRj2kHNwcYbpcAs8ToCfbuf7PPMqaErkvKvc4OvLi5t/XZ/fzF/9fDM/u3w9D51jEL56eXH5w9v59Vt4S61DzlydX/GLS35+8Xr+kx4GLa/P356++sec4/2POPrygp+9mZ/9Hbv09gGL0VtIc8BOgFC2Pfv1MbiRffnYl2NArIBxdgT8nJ2yBQhlBToFcRRgmABFxT+kCFAoSxgobUzMlYwgxipN0E6rqEwTw3uRpOBZiKTymRAZnCF48RGw+Agu6moL86/DDteoO65W7kKfAIDB5vb4zvsWZJku0zyE6BPVEKDX/pSTlXA+9WkvE+21ww9TTUc2cSxEIpLpCUGJqaIxPVH//hR4IcHhTU8w5Pco6zdIfDr1vI8T8RiLsmZz+oM3gPUYBvADcxIY8acCFZfX21JMT/DXFV7AObo0zvVbtQDh3Z68PDq6+zjRsUQrmQLB0X0mQmRJ9+grQVGruvXvIym41Ww9e5OySvPanb69/OH6bM7P/3l1eX0Dmvfj/OLm8vrncPoFiiZImnUpdTiDCAUiJImiC2uBYHAjcLsAsV+nFWgXap2e1dHRZNlEVcJJobjjIbvo3nK4gAwRFAww4mCLEKhfUbiW9PsT/oIqElkzI/tC0T6EFelyZjGMR9005oGg8yEJlcWAfYS2jRA07D1L8qGobgTgbLfx1ORj2LZDsC8HCBZHYGZEeJBSJIpleAPODRk2QBqkRzIEjFNm5IJVaCQa7sAfo8eFdqCAgQ0FCy59j1I4Gq8imsNZLUgErvoIXSx6b5rnae/9HXkZ0EJRgfDqNt3qnBdbRKBfSevkNJQjzcyKKJFqKvDcJvaHINJjD50EDVG2DUO0ERFcVY09orfaM0luHCRXmuvcweD7oshcTQqTAoJ46rmDch6iQjN6QBt6REgTtrUhiuaNUlpo+qxrs6xUw3KD1aiDdvD3ACYgfPJoAyxC47CDtn6JPhdgdlPWsAEw/ybNEr4Gz6aMGO/Ap0S4Hg2Dyi3I9R1sIaiLdeYYYZueY2DEZIHQH9PTVsOLfJEugbR+oeXVUqpElPAaMgxXoAJB+Aidpl7M/up4nYLSihHdEDElPWqbgWMA9wPC/vCxh9Xpre6omDBD2OYM0MwIlMhndSWEntIwsBK/NuDTJFtHeRNl8LxJxYMFaTpO2zPTEvUrWKO+M3mRc6KU9MgkfxqNuS0YRZkRMBxj+O7aLwoGhp9BWpdQ2EV1YFqQrez0olWc1VerPLD+fkEhWKSgXBK02TWrtx2eWriKgq5CYbQW31qXNz4P+TRbrTV9v+tlqgkiFxXZjdmDreN69IfSskB0i2g++wA+aQi8b5PHnlK0+S11WWbFveu8CMRyOUvzRaFGl+1Q8gTID2rS6NcFcKAWL3KJDl55kQEIe+Fj+amOskx5+M71f/1S7468MSF7fBWsC5ipyNNYL9n4GQyZh/I15Q0E7EzAqhcLnQerPg62qOz3nipkeP9mfvraabG4NdRy9s/JSHPUndqAQZrJOEXyFyH74MTgBFC8TR07J23pK8iLB9dUv0AZY7SCAtYBieZuZLVqGCd2FclRHIBGw3Vn6NpV/9Zl71CWcVFiL+cUK40zqhdqb9fVJn2dSJVRColgUSEAgpBEJcNlFSUQ1dCHbKIMa4Q7kN/JRY074IAKE4EJEXTbclLpBuUJeAJNIoF1KF0ZEtCaRMLnEdV5aqs7c5QMoeGpnAB0V4tdyejWDL07EB4tXaBBBqfCouoGIEiUoaffckNAjwNrX0DED4czeX2XqDc3nOTWUeQplhqMCf31LLD9PpmR/fRD8uEJeiyGwNxIGJiDORoiFoYxM9sFvV0Ph/FiPIDvYaeK3XpM2BLsNXekwvauFxyRFe2bQUAc2XafeJO3Q7mu1Dstif5+qcUUb8K9fnxQuFuvI1Wu2QX9awL6aYl/WhjKb+UF+OZEPO6m0q19UCcVkZW8QIjtYDA5lUoIuwZ9N8IaI4hxc9MIvvdS7yjU//3t9sKwucaTID1+mJvY2can1sp7K9GhR/+PpB77mQEiAY+Q1hyjQjhICZ5aAA5WaQuXq+irl9+E+ogjUI+uldh4wUo8JulSyJEQsIcwJEFPEIYePcKjdbLBrvZUxbT5EENUig+2QmZkTbY3PbudHX91dHRyt4fdB7yRTpDGMjLKSU1v4xLNpbGbpWo3Sv7zxxIAbzJWbHxiKVqB+vN8zq7FGmIlK/Jsy3LxoFzWrPUJXRYS3Us8lqCCUl2h1zG1pKDvKVDxhVXYNRfhNwD7COH2+Rw2az3TwXoq0vnjSqkIBtoOqvJLR0kduHxusfX7Xb4RP5ryOYVXw9YeCszkjEp5yilSnZr2PthDEEleFjJ93GMnmvZYTcC8UrY8vtWdgxtzqZPJoFpjejaoFvcIKLUwZeVP3ITRbtBQvqs7IFFU4oS0XhLCcPV842QI3FDvJ0HXsPK/a2ehbcXPMkzXYCJdh9yIKl2k2k8M1ngA2z0rgvTcjPIESZoQGpMiA4KGnNopzfrpuYhZtFYlIgCQRB3KK7l0lIdHdZSqWa+VHuIrO79Rrb3ZQH1KxHoS/HaeKN5WeFbgDrMy8Cs6ZfPZ1236ZGPPfSt/AttiKYDrztzQ0BztrZVqzVLLf4cUJsT7sfg+ddm1svaKqDpyeGv9k0bqrLPjdZTmLpbz7ZMHOo2tUJL6E4ngtFo2azCfK3rjJkLGVUoV95DzpIg596yRePzKIz3EdewzT8iwt6UIsXB3cIROGn1T+Eksp79niEr0MaGJaWHAc1SpGsY5Bwdqk5mZuKnXCLmEz4BHEZ4kfE2l4CIFyYYVctU99tk3x9p54YcSFKuIPv3hDJI4O1LuoVpIOPYpBI4K2sdhEgJjxsoZNMj4M1PQoEbVuytrUKPuYXCCKWSMnKRgqBuaBRUMtDpDSq7P4JCXnACrQUKozNoS9mTMeDndOQx0UwcxVLPqDmN0H1MloLe3J3+hIxmSKp2fDErl3/9wev2aziB6pye4AZ8CB38vttI+79fWcdRz/a0pk3P9aoLXdDo9pdAG8bsu6MMlgEq/idz+6onOWb7tPmHZRBmeCiNjLY4RGAusb5IAHYtHbZrdkOcWr+C2eNCdOcbdvVWtg+o01CT97w8zlH5AsJZr7ZC84IePz+jJkRfUvStMgijcnU2Rx9Nrepb3w06Hc2xyyk9n2c/I5D6tRGF/b/BJlUS8dC/tS+zyqv54isiNweCuDEtqjU1ddSJfQdBR63jq86J2EH6Gh6UUNFBrWR6Yn/XxETx11NOF9aCgi0VQnfNbmbX95tZeCCoMTd8tRgfNZ2C+weE7PAw97q2jI0BPVS1gMNSlveNNrIeUQ+OkTs8/xTrQZR0a1O5ZsUYzw64+mbZnFRO7WYYFRePbjG3uWXiLYcDvKnf6ZFVknFL4wZb7CQn94yEyBjpw/ArUVIHa4+mOFoUJ8rzBr00BOyTSVl6yW1T7XRxpA6Thh1Xi+XMwmnNG3+3qMwDzLS2B8ZwV1CfKumBFX83aTvz/GNxR+JipnasTyP8eQPxf4MN9qOFJjNjT710kQV13mv88ZHkAWHa48nmwchRVGlD5B2HK3wMp4a1ZFtUYOEeL51yXGFTt6S2ds88fIVEnfwBT/AdQSwECFAAUAAAACAAAAEldR4hv++ECAAC7BQAACgAAAAAAAAAAAAAApIEAAAAAYWdlbnQueWFtbFBLAQIUABQAAAAIAAAASV2YVzEmVAEAAO8CAAAVAAAAAAAAAAAAAACkgQkDAABjb25maWdzL3NhbXBsaW5nLnlhbWxQSwECFAAUAAAACAAAAEld6fUCsEwAAABfAAAAEAAAAAAAAAAAAAAApIGQBAAAZXZhbF9jb25maWcueWFtbFBLAQIUABQAAAAIAAAASV085LSLDgUAAJMJAAARAAAAAAAAAAAAAACkgQoFAABwcm9tcHRzL3N5c3RlbS5tZFBLAQIUABQAAAAIAAAASV2aRYnomQMAAGwGAAAfAAAAAAAAAAAAAACkgUcKAABza2lsbHMvcmVwYWlyLXdvcmtmbG93L1NLSUxMLm1kUEsBAhQAFAAAAAgAAABJXQnMyi4tFAAAQjsAACoAAAAAAAAAAAAAAKSBHQ4AAHNraWxscy9yZXBhaXItd29ya2Zsb3cvc2NyaXB0cy93b3JrZmxvdy5weVBLAQIUABQAAAAIAAAASV1Kr/xcWgUAADQLAAAdAAAAAAAAAAAAAACkgZIiAABza2lsbHMvc291cmNlLWltcG9ydC9TS0lMTC5tZFBLAQIUABQAAAAIAAAASV3GfNkbpw4AALQvAAAzAAAAAAAAAAAAAACkgScoAABza2lsbHMvc291cmNlLWltcG9ydC9zY3JpcHRzL3NvdXJjZV9pbXBvcnRfZ3VhcmQucHlQSwUGAAAAAAgACABJAgAAHzcAAAAA", "sha256": "12675d72152b30a2e58ccc3e17c57f7297967168809976619873392c85cd9194"}}')
SOURCE_BINDING = json.loads('{"source_sha256": {"competition/source_guard_native_panel.py": "0dfb1e93f2dfcfe062603fbc1a87cfc8a3d6dcbd6ee1418db96ac0d4845f7839", "competition/source_import_guard.py": "430fdc3dc241952d57f6b57240943fff8ff3847172b537aef6b6e28357b83618", "competition/frozen_agent_unpack.py": "d90d71ba52a118cd374a02d15149f01bd7a49dd902ab85a905152b8b2a342062", "competition/requests_native_agent_import_probe.py": "bc832df93643631643fa6dc8e7f4f2bd15ee9a1f968d7d9f80345ef8e21a1dc9", "scripts/prepare_source_guard_native_panel.py": "38f236152bf26f8db024cb512abcefb34216209e92f5976e0a72c654a2489a7a", "scripts/prepare_source_guard_native_amendment.py": "1a49f6f8b414c4cc21366db25d620f3100717231dad281b9e74c84dcd02502b1", "competition/source_validation_extension.py": "11fc4a1b34af15752bb69e798f04e4d76f0f8fa30da4109ea9465f3364b79995", "scripts/build_source_validation_candidate.py": "33e97c0620d5336eaa3500bf07e262035ecbce1fe459407226c794d85605f1d9", "competition/source_validation_native_panel.py": "49f8194d9d377d46a6cbce6750850f2045459ab901c37a0d736810de5e731b72", "scripts/prepare_source_validation_native_panel.py": "51871058691fc313e5436167c69227c3421d415d01784d88ba216377c1b19d5c", "scripts/build_focused_repair_candidate.py": "ef8b336807a85a0df3e990a3a5f591ee6c1d7b3cd4fec5f41378832e7e8b5d88", "competition/focused_repair_prompt.md": "42373511dd04c010351edf8abddc9b3dafb535bc412529fbba60ddad100e9743", "scripts/prepare_focused_repair_native_probe.py": "ccc65f3aac563e4d4db942096f7661db49f5423fc026a387709301911f8a135d", "scripts/build_structured_repair_candidate.py": "d3e43ff7edd4a80b2c585a247b7f9999f511a81c1f5dd51682ea70a07ea89b71", "competition/repair_workspace_skill.py": "eb57fc65c85f45eec7cf009abcedfecb25110b7c1d4869a9a9994f51984244de", "competition/structured_repair_prompt.md": "c154754ab44b6b99dc558eded7a9e62a8863c0c4f2c8ffcf8ecf0179cabcc6e4", "competition/structured_repair_SKILL.md": "a0220ba6eba26e3caed179c1ed98aed97602b993ba86795f49c85efef55480fc", "tests/test_repair_workspace_skill.py": "da30ec5db5d24da84eeb711ac6b5226fc2e396fb8c1f9f0971406fa04991b1c7", "reports/competition_focus/repair_workspace_skill_checks_20261007.xml": "d19e52f0524bb499c1ee0a8b5702a294b0062a247c337347e4637a25c8da6092", "reports/competition_focus/overnight_terminal_audit_20261007.json": "ace5662d7a571dc2b54b89265a5552da56412e4026c6892eb1934ee03f7b0a06", "competition/repair_workspace_native_panel.py": "6e50b3908de6f2bb07a63a9cecb78d67f467e655935a90fbe897588ec3b81e65", "scripts/prepare_structured_repair_native_probe.py": "6674042c3c9b74d3df6e72eb18a60c255e61b21bd6d056a13d324e4b3ee7b61b", "scripts/build_structured_repair_revision.py": "f123a0e7a073170ab2ee41d08f82eb3626216fe2e8abdec6cb484ad0c6078e8d", "competition/repair_workspace_skill_v19.py": "d4c232d1301d8030c539702f5be2d1db83f5b0397a0cddd3e8d60dfaa6acd3d7", "competition/structured_repair_SKILL_v19.md": "f4c517b2335238d1f63d3982def2062ab3031440eb8bc5a3e2d715d854ac661c", "tests/test_repair_workspace_skill_v19.py": "259d073aeef1e06fd201de990e545a54ad6247a1e53dd8e9b848dd9e536c1e29", "reports/competition_focus/repair_workspace_skill_v19_checks_20261007.xml": "a3376973ac791c08913fc8988adedc82143fd0885ad0977b796beda8ccdc94f5", "reports/kaggle_runs/20261007_structured_repair_native_v1_terminal_1130/reconciliation.json": "6c9e19bf00a08f81bb6ccdd851f33df10d3ca6e94fbb939987b5d4c0e8a1786c", "reports/competition_focus/adk_skill_executor_source_20261007/google/adk/tools/skill_toolset.py": "fa3267855239bf6951ddbd40671b5385c6f60962ee4fbd5b9ac04b51c53a9fc6", "competition/repair_workspace_native_panel_v19.py": "ec8adb45f8fdc8d202496287fb7bfb74e3129a328fe59b706feee1e909971604", "scripts/prepare_structured_repair_native_revision.py": "71ac72c47302138aa16858a6e35be3d4bf6d58b7532aacad040f610ff92fc398", "scripts/build_simple_repair_candidate.py": "9c649579a127cc8d9f3b99891e2fd5ad122415e7d1eb2e28db88f743f8e90ef0", "competition/simple_repair_workspace_skill_v20.py": "9c3125cd9c3bfaf45896cc56dd0845cfd90013f1fc22f504b777e2c3b0bcccf8", "competition/simple_repair_prompt_v20.md": "a04e29114e1018d179c92a6186c66e02e2468c8cc614ddaaab5ec897d522b613", "competition/simple_repair_SKILL_v20.md": "3a765e1579f695a90e53deb0862093e84b214505d994caa020aeb392f0cadc5b", "tests/test_repair_workspace_skill_v20.py": "27293b4fd36b30b995df1117ad92665f488515a605f716caf40a3397c26da4ab", "reports/competition_focus/repair_workspace_skill_v20_checks_20261007.xml": "ccb1a42b10878a29907374d839aa8de22e43a076265ac9016f7447e38ad46bd7", "reports/competition_focus/structured_repair_terminal_audit_20261007.json": "cc5a45e4d451ccd2311a29951924417552cf73587baa5f900560659bd6578e2b", "reports/kaggle_runs/20261007_structured_repair_native_v2_terminal/reconciliation.json": "e982ad1931516a99e7b9933dfd0d3d1c64838343b2a373f7cb88c7cf5a998411", "competition/repair_workspace_native_panel_v20.py": "c668169ab8bab4a886114d142aa04be14ae839b068fc172c9dd194533948b2aa", "scripts/prepare_simple_repair_native_probe.py": "e50a3c6ffd29f39e17335a62073c9c498eee91f1a5b29a08081e866758f0e059", "scripts/build_hypothesis_pipeline_candidate.py": "c51f5f7e90adafcc89e8fc621dcc55a80303f6762fe0d099c5cde763b1a49a32", "competition/hypothesis_planner_v21.md": "85ea8411a708049eee9a3c9317c8045ede88244a4e29144fc4cfb572b1dad4aa", "reports/competition_focus/source_validation_candidate_preparation_20261006.json": "1364896b1c8bf946968a54b058e8cefd43650652a2dcd6b1db9251b543235358", "reports/competition_focus/simple_repair_protocol_diagnosis_20261007.json": "3b3d96cede6d5927cd980cced88fd8e09e2b61aefda0e811d64d209c4104b363", "reports/competition_focus/simple_repair_candidate_selection_20261007.json": "998408574928b8d6841055d2a23d683327b300d0a993faf3f303113628750528", "reports/kaggle_runs/20261005_compiler_source_collection/recovered_checkpoints/0006_4faf4cc7093c84a57ac5db091f9ad6187b7525fe84a30e088418eb8ce02d6433/organizer_python/adk_submission/schema.py": "6554de60cfa208dcb0b095f4236271795b2955adf87f3be3c40dfa3d62521c5d", "reports/kaggle_runs/20261005_compiler_source_collection/recovered_checkpoints/0006_4faf4cc7093c84a57ac5db091f9ad6187b7525fe84a30e088418eb8ce02d6433/organizer_python/adk_submission/builders/workflows.py": "d8d7ccf9a51d57daec58c7231e2a62621a3ecca269e71ee64c57062cadadf980", "reports/kaggle_runs/20261005_compiler_source_collection/recovered_checkpoints/0006_4faf4cc7093c84a57ac5db091f9ad6187b7525fe84a30e088418eb8ce02d6433/organizer_python/adk_submission/builders/sub_agents.py": "ba6162832b1ca4e023e2848c6391242c18cbe9dadcfd560a6ebef81108c35b5a", "reports/kaggle_runs/20261005_compiler_source_collection/recovered_checkpoints/0006_4faf4cc7093c84a57ac5db091f9ad6187b7525fe84a30e088418eb8ce02d6433/organizer_python/swegemma/harness/agent_runner.py": "dff7d2505340633c3fe50ba7073f2609fb2b41becd69c356d9af25cc4c1f98dd", "reports/kaggle_runs/20261005_compiler_source_collection/recovered_checkpoints/0006_4faf4cc7093c84a57ac5db091f9ad6187b7525fe84a30e088418eb8ce02d6433/organizer_python/swegemma/config.py": "f035916218f895796c4d39f1f9ec25284152a6df57cf6de1fea4931165d2349f", "reports/kaggle_runs/20261005_compiler_source_collection/recovered_checkpoints/0006_4faf4cc7093c84a57ac5db091f9ad6187b7525fe84a30e088418eb8ce02d6433/organizer_python/swegemma/models/discovery.py": "70ff2d1719f71cd4c0d747fecfcf5763c1fbbb983684cc2d68b6771e3a3997ca", "competition/hypothesis_pipeline_native_panel.py": "7835b44c617653ec35c97b80375c9c7efb7459a5a3a8c6eb954dd5c3e29a00f0", "scripts/prepare_hypothesis_pipeline_native_probe.py": "06df533d03206c0a9cfb30336c79d42f3465c463f0c96859ed3b2cf4796959be", "competition/native_search_scope_panel.py": "90a142207798d2687e56d4850604b63050913a7653b26d2a8be8c4e4824b1371", "scripts/prepare_native_search_scope_probe.py": "5e366c95e3da1e71d2d24edcca5887dc100799178e7906b25c0df017868b54b7", "tests/test_native_search_scope_evidence.py": "37711a541e7c1d02839aff675dec0500af1dbb72c31475c9c6228f109f2d9e59", "reports/competition_focus/native_search_scope_evidence_checks_20261007.xml": "a3adde9ba7f354eb6342177336fd58ba4b16df03307005e4eabc6a2b16762032", "reports/competition_focus/hypothesis_pipeline_candidate_preparation_20261007.json": "d0221d8ba0e4325548b057cea9efc595aa993fffed9ba76a86e99853ae81bf7a", "competition/search_scope_v22.md": "fe04583dd97352806322aa667c27bd522b565d8ee5b1de0b8ded58dbc21eede9", "reports/kaggle_runs/20261007_native_search_scope_terminal_1817/reconciliation.json": "1e8a6ce050e611b75e6be36772621bc3b1abd5bcbe21d564398fa7f35adf6962", "scripts/build_search_scope_candidate.py": "cf8d28d3e8e69d6bf7ef84c59f085becf4b6d5e1aa975816183c05781abeacfe", "scripts/reconcile_native_search_scope_probe.py": "c14d92065f23575a0e415ab095cb426b8180c3631825566d0a87cfa7e1c15397", "reports/competition_focus/hypothesis_pipeline_candidate_selection_20261007.json": "8300e917a95cf758a79f9ac3ed99fbbb128c2ec2e0db587331db6e3603bcef9b", "scripts/prepare_search_scope_candidate_native.py": "88e68c6aaefb7e1230d1327f61c2fb56bf8f029526973aeb3d1735b7d791c7d2", "reports/competition_focus/search_scope_package_checks_20261007.xml": "f4dd436aa2b9b3e9aba1dc9a675a3054e7668ef7dc6bc6e4f697f0e1f28ce586", "tests/test_search_scope_package_boundary.py": "3ce22d60c77ec829c8d435d139442e43f89763c5dfa1ceec97b6a2779110eff5", "scripts/build_reasoning_budget_candidate.py": "2a3c0abdaaa20cdac726051ab99aa0d54c271da224b805f4b4957814cbc418a8", "reports/competition_focus/search_scope_terminal_timing_diagnosis_20261008.json": "5760d067cf2f248d41fc6dbb40ff3882579bbb366886b083fbdef031f49f9625", "reports/competition_focus/search_scope_candidate_selection_20261008.json": "6847af2cb332eb5cbac78105886d49453d4bdc44aa0f225db7f6745ae81e1fa0", "reports/competition_focus/vllm0191_budget_primary_20261008/source_receipt.json": "3aafefdc480e4cb02541009b50aa2188049b81a50d3c6744e36b05d5581d537f", "reports/kaggle_runs/20261005_compiler_source_collection/recovered_checkpoints/0006_4faf4cc7093c84a57ac5db091f9ad6187b7525fe84a30e088418eb8ce02d6433/organizer_python/adk_submission/server.py": "d60b3fe34dd7f6e359b003db82b68434757c74b1030862bacda5cff6164f91eb", "reports/kaggle_runs/20261005_compiler_source_collection/recovered_checkpoints/0006_4faf4cc7093c84a57ac5db091f9ad6187b7525fe84a30e088418eb8ce02d6433/organizer_python/adk_submission/resolvers/generation.py": "ba8bf237261220d46843c8d71118faa387fc07f876f072756dff44d641dd1c14", "reports/competition_focus/vllm0191_budget_primary_20261008/vllm/config/reasoning.py": "8bc102f01f330fb2509b4fca6b5dc102eb86ccf791362e847045d76778565cb3", "reports/competition_focus/vllm0191_budget_primary_20261008/vllm/reasoning/gemma4_reasoning_parser.py": "64a1d3f05e22cee893ebc1f2507a150eaadf4d06e3efac5035c6ab94480a739e", "reports/competition_focus/vllm0191_budget_primary_20261008/vllm/entrypoints/openai/chat_completion/protocol.py": "1a2492bdb99e944ca177853496408a2a00d578bb2fa51c6e49241567d6601a82", "reports/competition_focus/vllm0191_budget_primary_20261008/vllm/v1/sample/logits_processor/builtin.py": "769038f9f91a613f2157556e74413466218e1ca9a5e9e166487a3660809d6e28", "scripts/prepare_reasoning_budget_native.py": "176d5b4fa0570a25c815c15d48f9f7dbd6119052e113a7f1ca78ad1c0ca68c09", "reports/competition_focus/reasoning_budget_candidate_preparation_20261008.json": "4d8d044d9fcc8fc225658ddbfede021d3846ab8d3f806a65684e1939540a8ece", "competition/native_generation_wire_panel.py": "657aedafca3ce6a836a4201aff86d07dba68ce3456c09273eacd5c68235dd841", "reports/competition_focus/reasoning_budget_wire_boundary_checks_20261008.xml": "4a29c562a8a78f3fda913da0771ccd07f10b09ab69bf78cec88941b7d41febd8", "tests/test_reasoning_budget_wire_boundary.py": "070db2e2903d8030387bc59e74b424b2d0cb4100b4a60eea84c5f6613a012f4f", "competition/hypothesis_pipeline_scoring_trial.py": "cfd7d04b88e351e98caf58f351fe295ab066d56bf79e5aba65f9967e1277aa7a", "scripts/build_compact_writer_candidate.py": "96aecd6d777652d7dec7d0f7f2758b30649c4128ee977551a1a8c43d0b621be4", "competition/compact_writer_protocol_v24.md": "620afbffc68cc79f68df0569724eec8169628ecd3594a582ff08aec2aaeb6323", "reports/competition_focus/reasoning_budget_full_terminal_diagnosis_20261008.json": "1f5841d365eb0a9b319fed161658ba80c8fb6fa544eeea85f3a5b39f78ade7f3", "reports/competition_focus/reasoning_budget_candidate_selection_20261008.json": "95abf1d3b20214ceb651debd5440325e29d71a7db9ad4e6563daf692694116e5", "reports/competition_focus/frozen_gemma_template_boundary_replay_20261008.json": "22df969291171d4af66c2f050debd117db3146929f2b537c1889a2ffaea68eef", "scripts/prepare_compact_writer_native.py": "d326e4580d73d72b8f9d1d12d686026e16da0444475c2b9e0271fa9cf8adffe0", "reports/competition_focus/compact_writer_candidate_preparation_20261008.json": "563cc11e16cc3fa656a80b8684676ed34791450fc74a6bc31a9f51a30f4400e8", "competition/native_compact_writer_wire_panel.py": "8b36f0b3ed9113a21084bd3f42dd77c97c30cb459523183d1010ac5f9ee8ec5c", "reports/competition_focus/compact_writer_boundary_checks_20261008.xml": "ad3892c1b8b60090321dcbc3c32b59199eea454cff03c10a7137454992b7f59f", "tests/test_compact_writer_package_boundary.py": "6d68248dbe5988fef103a686428d352089932b8d68ef70de93ce4b02c3d14b10", "scripts/reconcile_compact_writer_native.py": "8689d8303612c1bdd193e501ff2c3dac482c47e030d87685f8ee4478c3a9e153", "scripts/build_bounded_compact_writer_candidate.py": "2a6ddba184451e15f9c713d523b851d8e6cc90e8983e0f60782ca17e3de0f4a9", "competition/bounded_compact_writer_protocol_v25.md": "5d73a4447d222b7ffbf9dea4318de6fcc036dbd0a701fe8ba448af9d30cef74c", "reports/competition_focus/stalled_prompt_reproduction_replay_20261008.json": "f915b6e90377ddc809f35987b207294c554e46bc6fe6e541aff9c498fffbb1c4", "reports/kaggle_runs/20261008_compact_writer_native_terminal_0653/reconciliation.json": "c1a3493c0d980ddad5ec70161eac046830745ee455cba9126efb7d9fb14fcad6", "scripts/prepare_bounded_writer_native.py": "c3a38c582c44a94053c17a0ec2ed82cf56e8da05b25ec44d03662fcd598171b0", "reports/competition_focus/bounded_compact_writer_candidate_preparation_20261008.json": "2558369f7a044ea7212c02d74a2299eba2a71e34d22e0f348c1860cd92b90977", "competition/native_bounded_writer_wire_panel.py": "7e2ea453342342afa33b8fdf6238f304f404fc564c5d926c2f447503e43ecbc0", "reports/competition_focus/bounded_writer_boundary_checks_20261008.xml": "31321b0223fd051e9e7811e4e70b6a442ce0515ceb367533e3e72ee1bd8c382b", "tests/test_bounded_writer_package_boundary.py": "d200de2077864fe87429a86bcef8c87e2f70218c43577eb5ea24d5c077660084", "scripts/reconcile_bounded_writer_native.py": "1cd84bc7cc8a95310234501b06fb896e1b7fa217da04b196588005405c63ab86", "scripts/build_edit_first_candidate.py": "6007ad54aea350e1611dd9d70408eabcb150eae1823d97d83ee036122417b450", "NEXT_PLAN.md": "5191fe9aeb6953ef5c52fda50fb047e7cea27ddd19882b4a67d91c5a5b52e906", "reports/competition_focus/next_plan_execution_v1/failure_census.json": "6844e9cdb74a9fc5cbee919782b8463aa05f9b2ff99182bbd23e986719162b0b", "competition/edit_first_workflow_v26.md": "67ec5bd807d6dc6b27a96bd24bcde692fe5dd795e1c8262a226e6101335c3262", "competition/edit_first_workflow_SKILL_v26.md": "97ec6d673d5040d65c65ce5c7d7f9e9d4b297698ac9c584b780d81f8c9434539", "competition/repository_workflow_skill.py": "0fa9731560c1fba22d805b45efed9c5c45f565c8fb89d695ef53f0504b384172", "tests/test_repository_workflow_skill.py": "8f64656a09678a4abe51f6fe042ce173ad60aa04f75a6a1026def97e752a49ff", "reports/competition_focus/structured_repair_candidate_preparation_20261007.json": "a92fcc5f764894d7da945068bcf586cea423456e29e284d94981e2e94a702970", "reports/competition_focus/simple_repair_candidate_preparation_20261007.json": "ef12d9e36354bf628ca0fad67a151b363a8432c5f85124deed9663e323e9b2ab", "scripts/prepare_edit_first_native.py": "ce6f4790f22a8b74d28919dd80a12b734d093bef32afa605396c8a394148129b", "competition/native_edit_first_skill_panel.py": "bdf653c2628ec5da16f213e726098602d7c44f2b05c8672f8b60eb4923ef75a5", "reports/competition_focus/next_plan_execution_v1/edit_first_candidate_preparation.json": "0eeca48e34aebf801f7bc06e1eb0d1702009b2c2f9c00fc5d06a3ec51f5dde06", "reports/competition_focus/bounded_writer_native_preparation_20261008.json": "5334e6db6d11bedc17d51c2dc314ffaeab15e368fcb083542654559089abd263", "scripts/prepare_edit_first_native_v2.py": "8dfa9e8365f233a877632e5bb657d3f93432e54210f5c8091982ea3029fe9cc9", "competition/native_edit_first_skill_panel_v2.py": "92b608fb77039c28e68eabc1d4c49adbeb5b3a9a29a6f75d28d883165bded321", "reports/competition_focus/next_plan_execution_v1/organizer_update_review.json": "b52df051b318fec7e967e680071d1691f059fc7828fcb1b2a95233432263e9b3", "reports/competition_focus/next_plan_execution_v1/current_organizer_capture/capture_receipt.json": "c905b9c707692dc234339f8d315cb5f1f08f72ff53675f12c7571f5dc77b2a96", "reports/kaggle_runs/20261009_edit_first_native_terminal_v1/reconciliation.json": "83e36dc83b687ff3ca6e8e81a386eedee59c5012c19dc73c7a651d047c3c6710", "scripts/build_production_first_candidate.py": "4dd5769a04ee2df774c7c6dae41702b9e299941db03cb3ec20e6c53a05ad4f1c", "reports/kaggle_runs/20261009_edit_first_native_terminal_v2/reconciliation.json": "4065a7bcfab94fbb820c72e8a020a445dc35554bf2976bee74a7f5eda312993a", "competition/repository_workflow_skill_v27.py": "c2ed1809220e65ee04ea84871446b83a203c8da74c89d165734e555afd073eba", "tests/test_repository_workflow_skill_v27.py": "00f54124df20975c6b0ed25291b68fcd29e6c8e37870d6a319ddc9d2023fcc06", "reports/competition_focus/next_plan_execution_v1/production_first_local_checks.log": "ecdbaa39bafccddaddfc776b3402d34ee68fe9f24b506f6c7db96935334285f7", "scripts/prepare_production_first_qualification.py": "b084eb93ed8130cb30b8a3df2d999ab525238b8879cf1e32bd6402af93b327e4", "reports/competition_focus/next_plan_execution_v1/production_first_candidate_preparation.json": "c1015d55646ad4b115b489fb0a53ef80475962926099cfad99f956d6cf783793", "competition/prospective_fixture_qualification.py": "a9f5a01b635e866608fc6a7020bab140ec5451365b5f5d287a540242e82458bb", "paper/feedback_utility_replay.py": "9cee4e1b5b5b40ed561e1d71120932f40944497d78a50de20b65189dccaa5186", "paper/feedback_utility_study_v2/plan.json": "8d936f292cb84bb87473a5fda000e43434d26cabbffad7fbab094b6ca71a8e79", "paper/feedback_utility_study_v2/snapshot_manifest.json": "328ac97986d053dda6b6af926c82635ac50036a3af956c50e226c975a3d3f003", "paper/feedback_utility_study_v2/windows_actual_v1.json": "30aaf62b02a2d489cd32a09ab14c0c448a407116daa147a3f4c84329b9801978", "reports/competition_focus/bounded_writer_comparison_preparation_20261008.json": "306663b0fcd9f774f2cdc66b91771454e04d87cd4311600737e58421170a495a", "benchmarks/competition/development/20261008_bounded_phase_repair_comparison_v1/plan.json": "2f1e887d70be2d35cf2efd63deb557e3fdb3770a1ee3dcbb28d2ba894c9bd81d", "scripts/prepare_production_first_comparison.py": "cf9c9962a47baf880aea5101bce58b8c7a0fc232169bd973d503bc96c21d6fa7", "reports/competition_focus/next_plan_execution_v1/production_first_qualification_review.json": "275d52a145a468428f7f96e3df24e41f875cafddc2894b49ed6aaf25c9e3bd38", "reports/competition_focus/next_plan_execution_v1/production_first_qualification_preparation.json": "120841460688bbadeb11913ff493dd0bb3f26417f1fbfa9db1e245e29f43736a", "competition/generation_only_workflow_trial.py": "3784942a0739e28b61b59eda610a1bb590875fa8d2acf83bea9d31236a2b7c48", "competition/generation_only_comparison.py": "cbc992a7d6b5103181bd437dd49b0fa888423bfcd5071ee15e35b4fd68c09b89", "tests/test_generation_only_comparison.py": "56f848a84a5b03ec3bf3222387d2b29b8fe16bfff2860f9077dbc54df7aa89e4"}, "plan_sha256": "c14a0d06b6eb85573aed9decf73bfe2e24b055345b62f5732923c099fdd567d0", "candidate_native_review_sha256": "275d52a145a468428f7f96e3df24e41f875cafddc2894b49ed6aaf25c9e3bd38", "problem_only_input_sha256": "711abc3a0494271c6e65c193f69f8a3f31b0e7eed0e2ed8105e6f64995c84453"}')
AGENT_INPUT_PACKET_BASE64 = 'eyJiYXNlX2NvbW1pdCI6ICJjZmYyMjM2ZGFjYjdhZWRjYWYyZDJkZTA4NzNjZmJjOWJiMmI4OTdhIiwgImhpbnRzX3RleHQiOiAiIiwgImluc3RhbmNlX2lkIjogImZhc3RhcGlfMTQ0MzAiLCAicHJvYmxlbV9zdGF0ZW1lbnQiOiAi8J+QmyBGaXggb3B0aW9uYWwgc2VxdWVuY2UgaGFuZGxpbmcgd2l0aCBuZXcgdW5pb24gc3ludGF4IGZyb20gUHl0aG9uIDMuMTBcblxuaHR0cHM6Ly9naXRodWIuY29tL2Zhc3RhcGkvZmFzdGFwaS9wdWxsLzE0Mjk3IGlzIG9ubHkgdGFraW5nIG9uZSB1bmlvbiB0eXBlIGludG8gYWNjb3VudC4gSSBjYW4gb25seSByZWNvbW1lbmQgdXNpbmcgaHR0cHM6Ly9naXRodWIuY29tL3B5ZGFudGljL3R5cGluZy1pbnNwZWN0aW9uIGZvciB0aGVzZSBraW5kIG9mIGNoZWNrcyAoc2VlIGh0dHBzOi8vdHlwaW5nLWluc3BlY3Rpb24ucHlkYW50aWMuZGV2L2xhdGVzdC91c2FnZS8gZm9yIG1vdGl2YXRpb24pLiIsICJyZXBvIjogImZhc3RhcGkvZmFzdGFwaSJ9CnsiYmFzZV9jb21taXQiOiAiZDJjMTdiNjAzZDJmOTg2ZDVmMzgwMGQzYmYyNGFjYmE4ZTFmNDg1YiIsICJoaW50c190ZXh0IjogIiIsICJpbnN0YW5jZV9pZCI6ICJmYXN0YXBpXzE0OTUzIiwgInByb2JsZW1fc3RhdGVtZW50IjogIuKZu++4jyBGaXggSlNPTiBTY2hlbWEgZm9yIGJ5dGVzLCB1c2UgYFwiY29udGVudE1lZGlhVHlwZVwiOiBcImFwcGxpY2F0aW9uL29jdGV0LXN0cmVhbVwiYCBpbnN0ZWFkIG9mIGBcImZvcm1hdFwiOiBcImJpbmFyeVwiYFxuXG7imbvvuI8gRml4IEpTT04gU2NoZW1hIGZvciBieXRlcywgdXNlIGBcImNvbnRlbnRNZWRpYVR5cGVcIjogXCJhcHBsaWNhdGlvbi9vY3RldC1zdHJlYW1cImAgaW5zdGVhZCBvZiBgXCJmb3JtYXRcIjogXCJiaW5hcnlcImBcclxuXHJcbiMjIEJhY2tncm91bmRcclxuXHJcbmBmb3JtYXQ6IGJpbmFyeWAgd2FzIGRlZmluZWQgaW4gT3BlbkFQSSAzLjAueCwgaW4gT3BlbkFQSSAzLjEueCB0aGUgc2NoZW1hIHdhcyBhbGlnbmVkIHdpdGggdGhlIGxhdGVzdCBKU09OIFNjaGVtYSwgcmVjb21tZW5kaW5nIGluc3RlYWQgYGNvbnRlbnRNZWRpYVR5cGU6IGFwcGxpY2F0aW9uL29jdGV0LXN0cmVhbWAuXHJcblxyXG5JIHN1c3BlY3QgdGhlIEpTT04gU2NoZW1hIGZvciBgYnl0ZXNgIHVzaW5nIGBcImZvcm1hdFwiOiBcImJpbmFyeVwiYCBjb21lcyBmcm9tIG15IGZpcnN0IGltcGxlbWVudGF0aW9uIGluIFB5ZGFudGljIDEueC5cclxuXHJcbkl0IHdhcyBkZWZpbmVkIGFuZCBzdWdnZXN0ZWQgaW4gT3BlbkFQSSAzLjAueCAobm90IGluIEpTT04gU2NoZW1hKTogaHR0cHM6Ly9zcGVjLm9wZW5hcGlzLm9yZy9vYXMvdjMuMC4zLmh0bWwjY29uc2lkZXJhdGlvbnMtZm9yLWZpbGUtdXBsb2Fkc1xyXG5cclxuT3BlbkFQSSAzLjEueCBhbGlnbmVkIHN1cHBvcnQgd2l0aCBKU09OIFNjaGVtYSBkcmFmdCAwNywgc28gaXQgd2FzIHN1Z2dlc3RlZCB0byB1cGF0ZSBmaWxlIHVwbG9hZHMgdG8gdXNlIHRoZSByZWd1bGFyIEpTT04gU2NoZW1hIGZvcm1hdDogYFwiY29udGVudE1lZGlhVHlwZVwiOiBcImFwcGxpY2F0aW9uL29jdGV0LXN0cmVhbVwiYDogaHR0cHM6Ly9sZWFybi5vcGVuYXBpcy5vcmcvdXBncmFkaW5nL3YzLjAtdG8tdjMuMVxyXG5cclxuVGhpcyBpcyBkZWZpbmVkIGluIEpTT04gU2NoZW1hIDA3OiBodHRwczovL2pzb24tc2NoZW1hLm9yZy9kcmFmdC0wNy9qc29uLXNjaGVtYS12YWxpZGF0aW9uI3JmYy5zZWN0aW9uLjguNFxyXG5cclxuIyMjIEpTT04gU2NoZW1hIDIwMjAtMTIgTm90ZVxyXG5cclxuTm93IE9wZW5BUEkgMy4yIGlzIGFsaWduZWQgd2l0aCBKU09OIFNjaGVtYSAyMDIwLTEyLCB3aGljaCBpcyB3aGF0IFB5ZGFudGljIHYyIGltcGxlbWVudHMgKGV4Y2VwdCBmb3IgdGhpcywgSSdtIGltcGxlbWVudGluZyBpdCB0aGVyZSB0b28pLlxyXG5cclxuSXQncyB0aGUgc2FtZSBhcyBpbiBKU09OIFNjaGVtYSBkcmFmdCAwNywgc28gdGhpcyBzdGlsbCBhcHBsaWVzOiBodHRwczovL2pzb24tc2NoZW1hLm9yZy9kcmFmdC8yMDIwLTEyL2RyYWZ0LWJodXR0b24tanNvbi1zY2hlbWEtdmFsaWRhdGlvbi0wMSNuYW1lLWNvbnRlbnRtZWRpYXR5cGVcclxuXHJcbiMjIyBVc2FnZSBpbiBKU09OXHJcblxyXG5KU09OIGFzIGEgZm9ybWF0IGFjdHVhbGx5IGRvZXNuJ3Qgc3VwcG9ydCBieXRlcywgZXZlcnl0aGluZyBoYXMgdG8gYmUgaW4gVVRGLTggc3RyaW5ncy4gVHJhbnNwb3J0aW5nIGJ5dGVzIGluIEpTT04gd291bGQgcmVxdWlyZSBlbmNvZGluZyBieXRlcyBpbiBhIHN0cmluZywgZS5nLiB3aXRoIGJhc2U2NC5cclxuXHJcbkJ1dCBhcyBKU09OIFNjaGVtYSBpcyBub3Qgb25seSBkZWZpbmVkIHRvIGRlY2xhcmUgSlNPTiBwYXlsb2FkcyBidXQgYWxzbyBwYXlsb2FkcyB0aGF0IGNvdWxkIGhhdmUgYSBjb21wYXJhYmxlIHN0cnVjdHVyZSBhbmQgZGVmaW5lZCB3aXRoIEpTT04gU2NoZW1hLCBpdCdzIHN0aWxsIHRoZXJlIGluIHRoZSBzcGVjLiIsICJyZXBvIjogImZhc3RhcGkvZmFzdGFwaSJ9CnsiYmFzZV9jb21taXQiOiAiNzFhMTdiNTkzMmUzOGJjZjc5OGZjZDA1YzViYjM4NGU2ZDRkYjViZiIsICJoaW50c190ZXh0IjogIiIsICJpbnN0YW5jZV9pZCI6ICJmYXN0YXBpXzE0NDgyIiwgInByb2JsZW1fc3RhdGVtZW50IjogIvCfkJsgRml4IGhhbmRsaW5nIGFyYml0cmFyeSB0eXBlcyB3aGVuIHVzaW5nIGBhcmJpdHJhcnlfdHlwZXNfYWxsb3dlZD1UcnVlYFxuXG7wn5CbIEZpeCBoYW5kbGluZyBhcmJpdHJhcnkgdHlwZXMgd2hlbiB1c2luZyBgYXJiaXRyYXJ5X3R5cGVzX2FsbG93ZWQ9VHJ1ZWBcclxuXHJcblJlbGF0ZWQgdG8gaHR0cHM6Ly9naXRodWIuY29tL2Zhc3RhcGkvZmFzdGFwaS9kaXNjdXNzaW9ucy8xNDE4NCwgaHR0cHM6Ly9naXRodWIuY29tL2Zhc3RhcGkvZmFzdGFwaS9pc3N1ZXMvMTQ0ODNcclxuXHJcbiIsICJyZXBvIjogImZhc3RhcGkvZmFzdGFwaSJ9CnsiYmFzZV9jb21taXQiOiAiNTI2NWM0ZjVjYjRkOGZiNGZiNTAxNWJjY2YxYTZkZThmM2ZjNjA0NCIsICJoaW50c190ZXh0IjogIiIsICJpbnN0YW5jZV9pZCI6ICJmYXN0YXBpXzE0MzYxIiwgInByb2JsZW1fc3RhdGVtZW50IjogIvCfkJsgQXZvaWQgYWNjZXNzaW5nIG5vbi1leGlzdGluZyBcIiRyZWZcIiBrZXkgZm9yIFB5ZGFudGljIHYyIGNvbXBhdCByZW1hcHBpbmdcblxuY2YgRGlzY3Vzc2lvbiAjMTQyNjUiLCAicmVwbyI6ICJmYXN0YXBpL2Zhc3RhcGkifQp7ImJhc2VfY29tbWl0IjogIjRkNmQ2MzFhM2QyZGVkZGY4NDA1NTIyZDRiOGM5NzZhNmQzNTcyNmMiLCAiaGludHNfdGV4dCI6ICIiLCAiaW5zdGFuY2VfaWQiOiAicmljaF8zODgyIiwgInByb2JsZW1fc3RhdGVtZW50IjogIkZpeCByYXcgbWFya3VwIHByaW50ZWQgb24gcHJvbXB0IGVycm9ycyBzb21ldGltZXNcblxuIyMgVHlwZSBvZiBjaGFuZ2VzXHJcblxyXG4tIFt4XSBCdWcgZml4XHJcbi0gWyBdIE5ldyBmZWF0dXJlXHJcbi0gWyBdIERvY3VtZW50YXRpb24gLyBkb2NzdHJpbmdzXHJcbi0gWyBdIFRlc3RzXHJcbi0gWyBdIE90aGVyXHJcblxyXG4jIyBDaGVja2xpc3RcclxuXHJcbi0gW3hdIEkndmUgcnVuIHRoZSBsYXRlc3QgW2JsYWNrXShodHRwczovL2dpdGh1Yi5jb20vcHNmL2JsYWNrKSB3aXRoIGRlZmF1bHQgYXJncyBvbiBuZXcgY29kZS5cclxuLSBbIF0gSSd2ZSB1cGRhdGVkIENIQU5HRUxPRy5tZCBhbmQgQ09OVFJJQlVUT1JTLm1kIHdoZXJlIGFwcHJvcHJpYXRlIChzZWUgbm90ZSBhYm91dCB0eXBvcyBhYm92ZSkuXHJcbi0gW3hdIEkndmUgYWRkZWQgdGVzdHMgZm9yIG5ldyBjb2RlLlxyXG4tIFt4XSBJIGFjY2VwdCB0aGF0IEB3aWxsbWNndWdhbiBtYXkgYmUgcGVkYW50aWMgaW4gdGhlIGNvZGUgcmV2aWV3LlxyXG5cclxuIyMgRGVzY3JpcHRpb25cclxuXHJcblByb21wdEJhc2Uub25fdmFsaWRhdGVfZXJyb3IoKSBzaG91bGQgZXhwbGljaXRseSBhc2sgZm9yIG1hcmt1cCBpbiBjYXNlIHRoZSBjb25zb2xlIHBhc3NlZCB0byBpdCBoYXMgbWFya3VwIGRpc2FibGVkIGdsb2JhbGx5LlxyXG5cclxuRml4ZXMgIzM4ODEuIiwgInJlcG8iOiAiVGV4dHVhbGl6ZS9yaWNoIn0KeyJiYXNlX2NvbW1pdCI6ICI0MTAxOTkxODk4ZWU3YTA5ZmUxNzA2ZGFjYTI0YWY1ZTFlMDU0ODYyIiwgImhpbnRzX3RleHQiOiAiIiwgImluc3RhbmNlX2lkIjogInJpY2hfMzUwNiIsICJwcm9ibGVtX3N0YXRlbWVudCI6ICJmaXggc3BsaXQgY2VsbHNcblxuRml4ZWQgYFNlZ21lbnQuX3NwbGl0X2NlbGxzYCBpc3N1ZSBkaXNjb3ZlcmVkIGluIGh0dHBzOi8vZ2l0aHViLmNvbS9UZXh0dWFsaXplL3RleHR1YWwvaXNzdWVzLzQ5OTYiLCAicmVwbyI6ICJUZXh0dWFsaXplL3JpY2gifQo='
if IS_KAGGLE and RUN_DECLARED_EXPERIMENT:
    RUN = Path('/kaggle/working') / ('production_first80_' + MODE + '_' + datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%f'))
    RUN.mkdir()
    frozen_plan_bytes = base64.b64decode(PLAN_BYTES_BASE64, validate=True)
    if hashlib.sha256(frozen_plan_bytes).hexdigest() != SOURCE_BINDING['plan_sha256'] or json.loads(frozen_plan_bytes) != PLAN:
        raise ValueError('Frozen plan bytes differ from source binding')
    (RUN / 'plan.json').write_bytes(frozen_plan_bytes)
    (RUN / 'source_binding.json').write_text(json.dumps(SOURCE_BINDING, indent=2) + '\n', encoding='utf-8')
    packet = base64.b64decode(AGENT_INPUT_PACKET_BASE64, validate=True)
    if hashlib.sha256(packet).hexdigest() != PLAN['selected_problem_only_input_sha256']:
        raise ValueError('Problem-only input differs from the frozen plan')
    (RUN / 'agent_inputs.jsonl').write_bytes(packet)
    unpack_frozen_agents(RUN, PLAN, PACKAGES)
    export_run(RUN)
    print('Frozen output directory:', RUN)


In [ ]:
"""Fail-fast, recorded CPU setup for the unchanged organizer source audit.

Definitions do not install packages. Runtime checks never filter incompatible
wheels into a superficially successful setup or establish scoring equivalence.
"""
from datetime import datetime, timezone
import glob
import hashlib
import importlib
import importlib.metadata
import json
import os
from pathlib import Path
import platform
import subprocess
import sys
import sysconfig
import traceback


REQUIRED_CPU_RUNTIME = {
    "implementation": "CPython", "python_major_minor": [3, 12],
    "system": "Linux", "machine": "x86_64",
}


def environment_file_sha256(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        while chunk := stream.read(1048576):
            digest.update(chunk)
    return digest.hexdigest()


def environment_package_versions(names):
    values = {}
    for name in sorted(names):
        try:
            values[name] = importlib.metadata.version(name)
        except importlib.metadata.PackageNotFoundError:
            values[name] = None
    return values


def cpu_runtime_snapshot(package_names):
    return {
        "implementation": platform.python_implementation(),
        "python_major_minor": list(sys.version_info[:2]),
        "python_version": sys.version, "executable": sys.executable,
        "executable_realpath": str(Path(sys.executable).resolve()),
        "system": platform.system(), "machine": platform.machine(),
        "platform": platform.platform(), "libc": list(platform.libc_ver()),
        "soabi": sysconfig.get_config_var("SOABI"),
        "multiarch": sysconfig.get_config_var("MULTIARCH"),
        "cuda_visible_devices": os.environ.get("CUDA_VISIBLE_DEVICES"),
        "package_versions": environment_package_versions(package_names),
    }


def cpu_runtime_errors(runtime):
    return [key + " must be " + repr(expected) + "; observed " + repr(runtime.get(key))
            for key, expected in REQUIRED_CPU_RUNTIME.items()
            if runtime.get(key) != expected]


def normalized_organizer_wheel_name(name):
    # Retain the organizer workaround already used in frozen notebook 10.
    return name.replace("cu128", "+cu128") if "cu128" in name and "+" not in name else name


def classify_organizer_wheels(paths, supported_tags=None):
    from packaging.tags import sys_tags
    from packaging.utils import parse_wheel_filename
    supported = set(sys_tags() if supported_tags is None else supported_tags)
    rows = []
    for path in sorted(map(Path, paths), key=lambda item: item.name):
        if not path.is_file():
            raise ValueError("Missing organizer wheel: " + str(path))
        row = {"source_path": str(path), "original_name": path.name,
               "install_name": normalized_organizer_wheel_name(path.name),
               "bytes": path.stat().st_size, "sha256": environment_file_sha256(path)}
        if "cutlass" in path.name.lower():
            row.update(selected=False, compatible=None,
                       exclusion_reason="existing_organizer_cutlass_workaround")
        else:
            try:
                distribution, version, build, tags = parse_wheel_filename(row["install_name"])
                row.update(selected=True, compatible=bool(tags & supported),
                           distribution=distribution, version=str(version),
                           wheel_tags=sorted(map(str, tags)),
                           matching_runtime_tags=sorted(map(str, tags & supported)))
            except ValueError as error:
                row.update(selected=True, compatible=False, wheel_parse_error=str(error))
        rows.append(row)
    if len({row["install_name"] for row in rows}) != len(rows):
        raise ValueError("Ambiguous organizer wheel aliases")
    return rows


def remove_recorded_cutlass_hooks():
    removed = []
    for pattern in ("/usr/local/lib/python*/dist-packages/*cutlass*.pth",
                    "/usr/local/lib/python*/site-packages/*cutlass*.pth"):
        for name in sorted(glob.glob(pattern)):
            path = Path(name)
            row = {"path": str(path), "sha256": environment_file_sha256(path)}
            path.unlink()
            removed.append(row)
    return removed


def source_audit_import_check():
    import torch
    from swegemma.config import EvalConfig
    from swegemma.evaluate import Evaluator
    from swegemma.models import load_tasks
    count = torch.cuda.device_count()
    if count != 0:
        raise ValueError("Select Accelerator None: source audit requires zero CUDA devices")
    return {"torch_version": torch.__version__, "cuda_device_count": count,
            "required_organizer_imports_succeeded": True}


def prepare_source_audit_environment(wheelhouse_dir, run_root, expected_packages,
                                     checkpoint_exporter, source_binding):
    """Install the whole selected organizer wheelhouse only after compatibility.

    Always retain runtime identity and setup failures before allowing the exact
    source inventory. UI labels are recorded separately from verified runtime.
    """
    run_root = Path(run_root)
    wheelhouse_dir = Path(wheelhouse_dir)
    run_root.mkdir(parents=True, exist_ok=False)
    package_names = set(expected_packages) | {"torch", "vllm", "pip", "packaging"}
    report = {
        "scope": "CPU_source_audit_environment_setup_not_model_or_grading",
        "created_utc": datetime.now(timezone.utc).isoformat(), "status": "started",
        "required_runtime": REQUIRED_CPU_RUNTIME, "source_binding": source_binding,
        "before": cpu_runtime_snapshot(package_names), "after": None,
        "wheelhouse_path": str(wheelhouse_dir), "wheel_manifest": [],
        "selected_wheels_omitted_for_incompatibility": [],
        "cutlass_hooks_removed": [], "pip_command": None, "pip_return_code": None,
        "model_inference_executed": False, "source_inventory_executed": False,
        "tests_executed": False, "private_scoring_equivalence_established": False,
        "runtime_image_id_verified": False,
        "requested_UI_profile": "original 2026-07-01 image, Accelerator None, Internet off",
    }

    def save():
        (run_root / "environment_receipt.json").write_text(
            json.dumps(report, indent=2) + "\n", encoding="utf-8")
        print("MOBILE_SOURCE_AUDIT_ENVIRONMENT=" + json.dumps({
            "status": report["status"], "receipt": str(run_root / "environment_receipt.json"),
            "python_version": report["before"]["python_version"],
            "runtime_errors": report.get("runtime_errors", []),
            "incompatible_wheels": report.get("incompatible_wheels", []),
            "pip_return_code": report["pip_return_code"],
        }), flush=True)
        checkpoint_exporter(run_root)

    save()
    try:
        if not wheelhouse_dir.is_dir():
            raise RuntimeError("Attach the official organizer wheelhouse before setup")
        report["wheel_manifest"] = classify_organizer_wheels(wheelhouse_dir.glob("*.whl"))
        selected = [row for row in report["wheel_manifest"] if row["selected"]]
        report["runtime_errors"] = cpu_runtime_errors(report["before"])
        report["incompatible_wheels"] = [row["original_name"] for row in selected if not row["compatible"]]
        if not selected or report["runtime_errors"] or report["incompatible_wheels"]:
            report["status"] = "preflight_rejected_before_install"
            raise RuntimeError("CPU source-audit preflight rejected this environment; "
                               "use a verified Linux x86_64 CPython 3.12 worker. "
                               "Incompatible organizer wheels were retained and no install ran.")
        report["status"] = "preflight_passed"
        save()
        links = run_root / "organizer_wheel_links"
        links.mkdir()
        for row in selected:
            (links / row["install_name"]).symlink_to(Path(row["source_path"]).resolve())
        report["cutlass_hooks_removed"] = remove_recorded_cutlass_hooks()
        command = [sys.executable, "-m", "pip", "install", "-q", "--no-index",
                   "--no-deps", "--force-reinstall"] + [str(links / row["install_name"]) for row in selected]
        report.update(status="installing", pip_command=command)
        save()
        result = subprocess.run(command, capture_output=True, text=True)
        (run_root / "organizer_install_stdout.txt").write_text(result.stdout or "", encoding="utf-8")
        (run_root / "organizer_install_stderr.txt").write_text(result.stderr or "", encoding="utf-8")
        report["pip_return_code"] = result.returncode
        importlib.invalidate_caches()
        report["after"] = cpu_runtime_snapshot(package_names)
        if result.returncode != 0:
            report["status"] = "organizer_install_failed"
            raise RuntimeError("Organizer wheel installation failed; inspect retained install logs")
        observed = {name: report["after"]["package_versions"][name] for name in expected_packages}
        if observed != expected_packages:
            report["status"] = "organizer_core_identity_rejected"
            raise ValueError("Installed organizer core packages differ from the frozen source-audit plan")
        report["status"] = "checking_CPU_organizer_imports"
        save()
        report.update(source_audit_import_check())
        report["status"] = "ready_for_unchanged_source_audit"
        save()
        return report
    except Exception as error:
        if report["status"] in {"started", "preflight_passed", "installing", "checking_CPU_organizer_imports"}:
            report["status"] = "setup_exception_before_source_audit"
        report.update(error_type=type(error).__name__, error=str(error))
        (run_root / "setup_exception.txt").write_text(traceback.format_exc(), encoding="utf-8")
        save()
        raise


In [ ]:
def resolve_organizer_audit_api(evaluator_module):
    verifier = getattr(evaluator_module, "verify_task", None)
    if not callable(verifier):
        raise ValueError("Installed Evaluator module has no callable official verify_task")
    implementation = inspect.getmodule(verifier)
    namespace = getattr(verifier, "__globals__", {})
    apis = {"verify_task": verifier}
    for name in ("resolve_task_snapshot_paths", "extract_snapshot", "sandbox_start", "sandbox_exec", "sandbox_stop"):
        # verify_task's own globals are the callables used by its retained source.
        # Snapshot resolution can be a public reexport from Evaluator's module.
        function = namespace.get(name)
        if function is None and implementation is not None:
            function = getattr(implementation, name, None)
        if function is None and name == "resolve_task_snapshot_paths":
            function = getattr(evaluator_module, name, None)
        if not callable(function):
            raise ValueError("Official verifier implementation lacks callable " + name)
        apis[name] = function
    placeholder = object()
    calls = {
        "resolve_task_snapshot_paths": ((placeholder, "instance_id", "repo"), {}),
        "extract_snapshot": ((placeholder, placeholder, placeholder),
                             {"patch_path": placeholder, "base_snapshot_path": placeholder}),
        "sandbox_start": ((placeholder,), {}),
        "sandbox_exec": ((placeholder, placeholder, "command"), {}),
        "sandbox_stop": ((placeholder, placeholder), {}),
    }
    for name, (args, kwargs) in calls.items():
        inspect.signature(apis[name]).bind(*args, **kwargs)
    for name in ("sandbox_start", "sandbox_exec", "sandbox_stop"):
        if not inspect.iscoroutinefunction(apis[name]):
            raise ValueError("Official sandbox callable is not asynchronous: " + name)
    return apis


In [ ]:
"""Recorded CPU/GPU preflight for separately frozen future public comparisons."""
from datetime import datetime, timezone
import hashlib
import importlib
import inspect
import json
from pathlib import Path
import subprocess
import sys
import tempfile
import traceback



def declared_backend_check(mode, torch):
    count = torch.cuda.device_count()
    names = [torch.cuda.get_device_name(index) for index in range(count)]
    if mode == 'cpu_controls':
        if count != 0:
            raise ValueError('Select Accelerator None for fixture controls')
    elif mode == 'gpu_comparison':
        if count != 4 or any('L4' not in name for name in names):
            raise ValueError('The declared comparison requires exactly four NVIDIA L4 GPUs')
    else:
        raise ValueError('Unknown declared experiment mode')
    return {'cuda_device_count': count, 'gpu_names': names, 'torch_version': torch.__version__}


def prepare_declared_environment(wheelhouse_dir, run_root, plan, checkpoint_exporter, mode):
    run_root, wheelhouse_dir = Path(run_root), Path(wheelhouse_dir)
    receipt = run_root / 'environment_receipt.json'
    if receipt.exists():
        raise FileExistsError('Preserve the existing environment attempt')
    expected = plan['expected_core_packages']
    names = set(expected) | {'torch', 'vllm', 'pip', 'packaging'}
    report = {'created_utc': datetime.now(timezone.utc).isoformat(),
              'scope': 'declared_public_comparison_environment_not_model_or_control_results',
              'mode': mode, 'status': 'started', 'before': cpu_runtime_snapshot(names),
              'after': None, 'wheel_manifest': [], 'incompatible_wheels': [],
              'selected_wheels_omitted_for_incompatibility': [], 'pip_return_code': None,
              'plan_sha256': environment_file_sha256(run_root / 'plan.json'),
              'model_inference_executed': False, 'fixture_controls_executed': False,
              'runtime_image_id_verified': False}
    def save():
        receipt.write_text(json.dumps(report, indent=2) + '\n', encoding='utf-8')
        print('DECLARED_ENVIRONMENT=' + json.dumps({key: report.get(key) for key in
            ('mode', 'status', 'runtime_errors', 'incompatible_wheels', 'pip_return_code')}), flush=True)
        checkpoint_exporter(run_root)
    save()
    try:
        if not wheelhouse_dir.is_dir():
            raise ValueError('Attach the complete official organizer wheelhouse')
        report['runtime_errors'] = cpu_runtime_errors(report['before'])
        report['wheel_manifest'] = classify_organizer_wheels(wheelhouse_dir.glob('*.whl'))
        selected = [row for row in report['wheel_manifest'] if row['selected']]
        report['incompatible_wheels'] = [row['original_name'] for row in selected if not row['compatible']]
        core_wheels = {r['distribution']: {'version':r['version'],'sha256':r['sha256']}
            for r in selected if r['distribution'] in plan['expected_core_wheels']}
        if core_wheels != plan['expected_core_wheels']:
            report['status'] = 'preflight_rejected_before_install'
            raise ValueError('Organizer wheels changed again; preserve this attempt before install')
        if not selected or report['runtime_errors'] or report['incompatible_wheels']:
            report['status'] = 'preflight_rejected_before_install'
            raise RuntimeError('Use a compatible Linux x86_64 CPython 3.12 organizer worker; no incompatible wheels were omitted or installed')
        report['cutlass_hooks_removed'] = remove_recorded_cutlass_hooks()
        # Link staging stays outside the exported run; its wheel identities remain
        # in the receipt. It cannot collide with strict artifact link rejection.
        with tempfile.TemporaryDirectory(prefix='neural_swe_declared_wheels_') as temporary:
            links = Path(temporary)
            for row in selected:
                (links / row['install_name']).symlink_to(Path(row['source_path']).resolve())
            command = [sys.executable, '-m', 'pip', 'install', '-q', '--no-index',
                       '--no-deps', '--force-reinstall'] + [str(links / row['install_name']) for row in selected]
            report.update(status='installing', pip_command=command)
            save()
            result = subprocess.run(command, capture_output=True, text=True, timeout=1200)
            (run_root / 'organizer_install_stdout.txt').write_text(result.stdout or '', encoding='utf-8')
            (run_root / 'organizer_install_stderr.txt').write_text(result.stderr or '', encoding='utf-8')
            report['pip_return_code'] = result.returncode
        importlib.invalidate_caches()
        report['after'] = cpu_runtime_snapshot(names)
        if result.returncode != 0:
            raise RuntimeError('Organizer installation failed; retained install logs are available')
        if {name: report['after']['package_versions'][name] for name in expected} != expected:
            raise ValueError('Installed organizer core identity differs from the frozen plan')
        import torch
        from swegemma.config import EvalConfig
        from swegemma.evaluate import Evaluator
        from swegemma.models import load_tasks
        report.update(declared_backend_check(mode, torch))
        apis = resolve_organizer_audit_api(inspect.getmodule(Evaluator))
        verifier_sha = hashlib.sha256(inspect.getsource(apis['verify_task']).encode()).hexdigest()
        report['verify_task_sha256'] = verifier_sha
        if verifier_sha != plan['expected_verify_task_sha256']:
            raise ValueError('Actual organizer verifier differs from the frozen contract')
        report['status'] = 'ready_for_' + mode
        save()
        return report
    except Exception as error:
        if report['status'] != 'preflight_rejected_before_install':
            report['status'] = 'setup_exception_before_experiment'
        report.update(error_type=type(error).__name__, error=str(error))
        (run_root / 'setup_exception.txt').write_text(traceback.format_exc(), encoding='utf-8')
        save()
        raise


In [ ]:
if IS_KAGGLE and RUN_DECLARED_EXPERIMENT:
    if MODE == 'gpu_comparison':
        if not MODEL_PATH.is_dir():
            raise ValueError('Attach exact 31B QAT revision 2 model')
        for key, value in {'LITELLM_LOCAL_MODEL_COST_MAP':'True', 'TRANSFORMERS_NO_TF':'1',
            'VLLM_WORKER_MULTIPROC_METHOD':'spawn', 'VLLM_MEMORY_PROFILER_ESTIMATE_CUDAGRAPHS':'1',
            'VLLM_ENGINE_READY_TIMEOUT_S':'1200', 'VLLM_NO_USAGE_STATS':'1',
            'OTEL_SDK_DISABLED':'true', 'PYTORCH_CUDA_ALLOC_CONF':'expandable_segments:True'}.items():
            os.environ[key] = value
    prepare_declared_environment(WHEELHOUSE_DIR, RUN, PLAN, export_run, MODE)
    if hashlib.sha256((DATA_DIR / 'tasks.jsonl').read_bytes()).hexdigest() != PLAN['tasks_sha256']:
        raise ValueError('Official task revision differs from the frozen plan')
    if hashlib.sha256((DATA_DIR / 'HARNESS_README.md').read_bytes()).hexdigest() != PLAN['guide_sha256']:
        raise ValueError('Official guide revision differs from the frozen plan')
    from swegemma.models import load_tasks
    actual = {task.instance_id: task for task in load_tasks(DATA_DIR / 'tasks.jsonl')}
    for line in (RUN / 'agent_inputs.jsonl').read_text(encoding='utf-8').splitlines():
        problem = json.loads(line)
        if set(problem) != {'instance_id', 'repo', 'base_commit', 'problem_statement', 'hints_text'}:
            raise ValueError('Problem-only input includes unexpected fields')
        if any(getattr(actual[problem['instance_id']], name) != value for name, value in problem.items()):
            raise ValueError('Actual organizer problem-only input differs from frozen input')


In [ ]:
"""A small, locked development comparison using the actual organizer evaluator.

Development results cannot be reported as held-out or leaderboard results. Every
assigned arm/task pair remains in the denominator, including interrupted runs.
"""
from datetime import datetime, timezone
import hashlib
import json
import os
from pathlib import Path
import re
import tempfile
import time


def assignment_key(row: dict) -> tuple:
    """Explicit trial IDs allow repeated assignments without merging outcomes."""
    values = (row['arm'], row['task_id'])
    for value in (*values, *([row['trial_id']] if 'trial_id' in row else [])):
        if not isinstance(value, str) or not re.fullmatch(r'[A-Za-z0-9_-]+', value):
            raise ValueError('Assignment identifiers must be safe nonempty path components')
    return (*values, row.get('trial_id'))


def atomic_json(path: Path, value: dict) -> None:
    """A crash must leave the previous complete receipt or the next complete one."""
    temporary = None
    try:
        with tempfile.NamedTemporaryFile(mode='w', encoding='utf-8', newline='\n',
                                         dir=path.parent, prefix='.' + path.name + '.',
                                         suffix='.tmp', delete=False) as stream:
            temporary = Path(stream.name)
            stream.write(json.dumps(value, indent=2) + '\n')
            stream.flush()
            os.fsync(stream.fileno())
        os.replace(temporary, path)
    finally:
        if temporary is not None and temporary.exists():
            temporary.unlink()


def summarize(plan: dict, observations: list[dict]) -> dict:
    if not plan['schedule']:
        raise ValueError('A locked comparison must assign at least one trial')
    for arm in plan['arms']:
        assignment_key({'arm': arm, 'task_id': 'configuration'})
    expected = {assignment_key(row) for row in plan['schedule']}
    if len(expected) != len(plan['schedule']):
        raise ValueError('Duplicate planned assignment; repeats require distinct trial_id values')
    if any(pair[0] not in plan['arms'] for pair in expected):
        raise ValueError('Scheduled arm is not declared')
    seen, counts = set(), {arm: 0 for arm in plan['arms']}
    completed = 0
    for row in observations:
        key = assignment_key(row)
        if key not in expected or key in seen:
            raise ValueError('Unexpected or duplicate observed arm/task pair')
        seen.add(key)
        result = row.get('result', {})
        completed += int(result.get('status') == 'completed')
        passed = (result.get('status') == 'completed'
                  and result.get('official_resolved') is True
                  and result.get('test_exit_code') == 0
                  and result.get('patch_bytes', 0) > 0)
        counts[row['arm']] += int(passed)
    return {
        'scope': 'public_development_comparison_not_held_out_or_leaderboard',
        'assigned_pairs': len(expected), 'observed_pairs': len(seen),
        'completed_pairs': completed, 'incomplete_observed_pairs': len(seen) - completed,
        'unattempted_pairs': len(expected - seen),
        'arms': {arm: {'verified_resolved': counts[arm],
                      'assigned_count': sum(pair[0] == arm for pair in expected)}
                 for arm in plan['arms']},
        'official_leaderboard_score': None,
        'general_superiority_established': False,
        'interpretation': 'Development outcomes and repeated trials do not establish held-out general superiority.',
    }


In [ ]:
# Separately declared CPU26-qualified scoring-compaction transfer. GPU25 is preserved.
"""One real development issue using the organizer's published starter API.

API provenance: Ryan Holbrook, Getting Started - Gemma 4 Developer Agent,
version 2 (353282641), accessed 2026-10-03, Apache-2.0.
https://www.kaggle.com/code/ryanholbrook/getting-started-gemma-4-developer-agent
No substitute model, mock grader or inferred leaderboard score is provided.
"""
from __future__ import annotations

from datetime import datetime, timezone
import hashlib
import importlib.metadata
import json
from pathlib import Path
import time


def validated_task_limits(value=None):
    defaults = {'timeout_seconds': 300, 'max_time_minutes': 30,
                'max_tool_calls': 50, 'max_turns': 100}
    if value is None:
        return defaults
    if (not isinstance(value, dict) or set(value) != set(defaults)
            or any(type(number) is not int or number <= 0 for number in value.values())):
        raise ValueError('Task limits must declare all four positive integer budgets')
    return dict(value)


def exact_budget_sidecar(agent_dir, task_limits):
    # Check the frozen canonical bytes, without pretending to execute Stage1.
    names = ('timeout_seconds', 'max_tool_calls', 'max_time_minutes', 'max_turns')
    expected = ('evaluation:\n' + ''.join('  ' + key + ': ' + str(task_limits[key]) + '\n' for key in names)).encode()
    actual = (Path(agent_dir) / 'eval_config.yaml').read_bytes()
    if actual != expected:
        raise ValueError('Budget sidecar bytes differ from explicit frozen task limits')
    return {'sha256': hashlib.sha256(actual).hexdigest(), 'bytes': len(actual),
            'canonical_sidecar_bytes_match_explicit_limits': True,
            'official_stage1_sidecar_loader_executed': False}


def validate_no_adapter_source_members(agent_dir):
    """Permit exactly the frozen plain baseline or prospectively frozen native-interface or sequential hypothesis package."""
    agent_dir = Path(agent_dir)
    paths = list(agent_dir.rglob('*'))
    if any(path.is_symlink() or not path.resolve().is_relative_to(agent_dir.resolve()) for path in paths):
        raise ValueError('Agent package contains a link or escaped path')
    required = {'agent.yaml', 'prompts/system.md', 'configs/sampling.yaml', 'eval_config.yaml'}
    skill = {'skills/source-import/SKILL.md', 'skills/source-import/scripts/source_import_guard.py'}
    pipeline = {'planner.yaml', 'writer.yaml', 'prompts/planner.md', 'configs/planner_sampling.yaml'}
    actual = {path.relative_to(agent_dir).as_posix() for path in paths if path.is_file()}
    workflow={'skills/repair-workflow/SKILL.md','skills/repair-workflow/scripts/workflow.py'}
    if actual not in (required | skill, required | skill | pipeline, required | skill | workflow):
        raise ValueError('Unexpected files in the frozen no-adapter hypothesis-pipeline study')
    return actual



def compiled_agent_tree(agent):
    row = dict(name=agent.name, agent_class=type(agent).__module__ + '.' + type(agent).__name__)
    if hasattr(agent, 'generate_content_config'):
        row.update(generation=agent.generate_content_config.model_dump(mode='json', exclude_none=True),
            tool_names=[getattr(t, 'name', getattr(t, '__name__', None)) for t in agent.tools],
            instruction_sha256=hashlib.sha256(agent.instruction.encode()).hexdigest(),
            include_contents=agent.include_contents, output_key=agent.output_key)
    row['children'] = [compiled_agent_tree(child) for child in agent.sub_agents]
    return row


def verify_compiled_hypothesis_package(tree, agent_dir):
    agent_dir = Path(agent_dir)
    # These files are frozen and verified against the study plan before runtime.
    import yaml
    expected_writer = yaml.safe_load((agent_dir / 'configs/sampling.yaml').read_bytes())
    native_tools = ['run_command','read_file','edit_file','write_file','get_status','submit_patch',
                    'search_similar_code','get_code_neighbors','get_code_subgraph',None]
    if (agent_dir / 'planner.yaml').exists():
        if not tree['agent_class'].endswith('.SequentialAgent') or len(tree['children']) != 2:
            raise ValueError('Native sequential structure differs')
        planner, writer = tree['children']
        expected_planner = yaml.safe_load((agent_dir / 'configs/planner_sampling.yaml').read_bytes())
        if (planner['generation'] != expected_planner or planner['tool_names'] != []
                or planner['include_contents'] != 'none' or planner['output_key'] != 'repair_hypothesis'
                or planner['instruction_sha256'] != hashlib.sha256((agent_dir / 'prompts/planner.md').read_text(encoding='utf-8').encode()).hexdigest()):
            raise ValueError('Native planner configuration differs')
    else:
        writer = tree
    if (writer['generation'] != expected_writer or writer['tool_names'] != native_tools
            or writer['instruction_sha256'] != hashlib.sha256((agent_dir / 'prompts/system.md').read_text(encoding='utf-8').encode()).hexdigest()):
        raise ValueError('Native writer configuration differs')


def run_generation_only_workflow_trial(data_dir: Path, model_path: Path, agent_dir: Path, results_dir: Path, task_id: str,
        *, runtime: dict | None = None, task_limits: dict | None = None) -> dict:
    task_limits = validated_task_limits(task_limits)
    sidecar_receipt = exact_budget_sidecar(agent_dir, task_limits)
    # Imports and API names below are from the organizer starter, not guesses.
    import asyncio
    import concurrent.futures
    import litellm
    import torch
    from adk_submission import VllmConfig, VllmServer, discover_adapters
    from google.adk.agents.context_cache_config import ContextCacheConfig
    from google.adk.apps._configs import EventsCompactionConfig
    from swegemma.config import ALLOWED_ADAPTER_EXTENSIONS, EvalConfig, build_submission_limits
    from swegemma.evaluate import Evaluator
    from swegemma.models import load_tasks
    from swegemma.models.discovery import validate_single_declared_model

    model = 'gemma-4-31b-it-qat-w4a16-ct'
    if not model_path.is_dir() or not (data_dir / 'snapshots').is_dir():
        raise ValueError('Attach the exact required model and complete official data')
    if not torch.cuda.is_available() or torch.cuda.device_count() != 4:
        raise ValueError('This smoke study requires the competition L4 x4 environment')
    gpu_names = [torch.cuda.get_device_name(i) for i in range(4)]
    if any('L4' not in name for name in gpu_names):
        raise ValueError('Unexpected GPU environment; no model/backend substitution')
    tasks = [task for task in load_tasks(data_dir / 'tasks.jsonl') if task.instance_id == task_id]
    if len(tasks) != 1: raise ValueError('Select exactly one real development task')
    declared = validate_single_declared_model(agent_dir)
    if declared != model: raise ValueError('Declared base model differs from competition requirement')
    adapters = discover_adapters(str(agent_dir), adapter_extensions=ALLOWED_ADAPTER_EXTENSIONS)
    # discover_adapters returns a manifest object, whose truth value does not
    # establish whether it contains adapters. This baseline is confined to the
    # exact plan-bound declarative files and optional audited source-import skill, so no adapter weights can enter the run.
    actual_files = validate_no_adapter_source_members(agent_dir)
    results_dir.mkdir(parents=True, exist_ok=False)
    report = {'scope': 'generation_only_public_development_issue_independent_CPU_grading_required',
              'created_utc': datetime.now(timezone.utc).isoformat(), 'assigned_ids': [task_id],
              'model': model, 'model_path': str(model_path), 'gpu_names': gpu_names,
              'evaluation_limits': task_limits,
              'guide_sha256': hashlib.sha256((data_dir / 'HARNESS_README.md').read_bytes()).hexdigest(),
              'source_sha256': hashlib.sha256((data_dir / 'tasks.jsonl').read_bytes()).hexdigest(),
              'packages': {name: importlib.metadata.version(name) for name in
                           ('swegemma', 'adk-submission', 'adk-eval-core', 'vllm')},
              'assigned_count': 1, 'verified_resolved': 0, 'status': 'started',
              'official_leaderboard_score': None}
    report_path = results_dir / 'smoke_outcome.json'
    def save(): report_path.write_text(json.dumps(report, indent=2) + '\n', encoding='utf-8')
    save()
    started = time.monotonic()
    try:
        litellm.drop_params = True
        runtime_was_reused = bool(runtime)
        runtime_setup_started = time.monotonic()
        if runtime is not None and runtime:
            if runtime['model_path'] != str(model_path):
                raise ValueError('Cannot reuse a runtime with another model')
            models = runtime['models']
        else:
            server = VllmServer(VllmConfig(model=str(model_path), port=8000, host='127.0.0.1',
            tool_call_parser='gemma4', reasoning_parser='gemma4', max_model_len=32768,
            dtype='bfloat16' if torch.cuda.is_bf16_supported() else 'auto',
            gpu_memory_utilization=.80, enable_auto_tool_choice=True, enable_lora=True,
            max_loras=8, max_lora_rank=128, tensor_parallel_size=4, startup_timeout=1200),
            adapter_manifest=adapters)
            server.start()
            models = server.create_model_registry(aliases=[declared], model_prefix='openai/', api_key='EMPTY')
            if runtime is not None:
                runtime.update(model_path=str(model_path), models=models, server=server)
        report['model_runtime_reused'] = runtime_was_reused
        report['model_runtime_setup_wall_seconds'] = time.monotonic() - runtime_setup_started
        limits, constraints = build_submission_limits()
        config = EvalConfig(tasks_path=data_dir / 'tasks.jsonl', snapshots_dir=data_dir / 'snapshots',
            results_dir=results_dir, submission_dir=agent_dir, models=models, sandbox='subprocess',
            **task_limits,
            limits=limits, generation_constraints=constraints, adapter_manifest=adapters,
            context_cache_config=ContextCacheConfig(min_tokens=2048, ttl_seconds=1800, cache_intervals=10),
            events_compaction_config=EventsCompactionConfig(compaction_interval=5, overlap_size=2,
                token_threshold=14336, event_retention_size=5),
            graph_dir=str(data_dir / 'graphs'), embeddings_dir=str(data_dir / 'embeddings'),
            wheels_dir=data_dir / 'wheels', verbose=False)
        effective = {
            'timeout_seconds': config.harness.command_timeout_seconds,
            'max_time_minutes': config.budget.time_minutes,
            'max_tool_calls': config.budget.tool_calls,
            'max_turns': config.budget.turns,
        }
        if effective != task_limits:
            raise ValueError('Actual constructed task budget differs from the frozen plan')
        compaction = {key: getattr(config.events_compaction_config, key) for key in
            ('compaction_interval', 'overlap_size', 'token_threshold', 'event_retention_size')}
        cache = {key: getattr(config.context_cache_config, key) for key in
            ('min_tokens', 'ttl_seconds', 'cache_intervals')}
        if compaction != dict(compaction_interval=5, overlap_size=2, token_threshold=14336, event_retention_size=5):
            raise ValueError('Actual compaction differs from the shared frozen setting')
        report['effective_configuration'] = {
            'task_limits': effective,
            'events_compaction': compaction,
            'context_cache': cache,
            'stage1_sidecar_loader_executed': False,
            'budget_entry_path': 'Explicit EvalConfig kwargs after canonical sidecar byte check; Stage1 loader not executed',
            'local_exact_sidecar_verification': sidecar_receipt,
            'timeout_scope': 'Per sandbox command including verifier pytest; no whole-verifier deadline',
        }
        save()
        evaluator = Evaluator(config)
        # A worker owns its asyncio loop, including inside a live notebook kernel.
        compilation = []
        def evaluate():
            # Read-only observation: return the original compiler's same object.
            # No state, callback, model client or tool behavior is replaced.
            from unittest.mock import patch
            import swegemma.harness.agent_runner as native_runner
            original_compile = native_runner.compile_submission
            def observe_compile(*args, **kwargs):
                agent = original_compile(*args, **kwargs)
                tree = compiled_agent_tree(agent)
                verify_compiled_hypothesis_package(tree, agent_dir)
                compilation.append(tree)
                report['native_compilation'] = compilation
                report['native_compilation_observation_changes_agent_behavior'] = False
                save()
                return agent
            from swegemma.context import SwegemmaContext
            from swegemma.harness.verification import save_trace_artifact
            context=SwegemmaContext(docker_manager=evaluator.docker,task=tasks[0],
                budget=config.budget,harness=config.harness,graph_dir=config.graph_dir,embeddings_dir=config.embeddings_dir)
            snapshot,base,task_patch=__import__('swegemma.evaluate',fromlist=['resolve_task_snapshot_paths']).resolve_task_snapshot_paths(
                data_dir/'snapshots',tasks[0].instance_id,tasks[0].repo)
            with patch.object(native_runner, 'compile_submission', observe_compile):
                patch_text,error,trace=asyncio.run(native_runner.run_agent_sandbox(evaluator.docker,config,tasks[0],snapshot,
                    base_snapshot_path=base,patch_path=task_patch,context=context))
            trace_path=save_trace_artifact(trace,results_dir,tasks[0].instance_id)
            return patch_text,error,context,trace_path
        with concurrent.futures.ThreadPoolExecutor(max_workers=1) as pool:
            result = pool.submit(evaluate).result()
        patch_text,error,context,trace_path=result
        patch=patch_text or ''
        (results_dir/'generated.patch').write_text(patch,encoding='utf-8')
        report.update(status='completed',generation_returned=True,native_agent_error=error,
            generation_completed_without_agent_error=error is None,official_resolved=None,test_exit_code=None,
            grader_executed=False,independent_CPU_validation_pending=True,
            trace_artifact=str(trace_path),tool_calls=context.tool_calls_used,
            charged_LLM_calls=context.llm_calls_used,charged_calls_are_not_completed_response_count=True,
            agent_elapsed_seconds=context.agent_elapsed_seconds,patch_bytes=len(patch.encode()),
            patch_sha256=hashlib.sha256(patch.encode()).hexdigest(),verified_resolved=0,
            actual_original_agent_runner_and_sandbox_used=True,synthetic_model_client_used=False,
            reference_fix_or_validation_instructions_passed_to_model=False)
    except Exception as error:
        report.update(status='failed', error_type=type(error).__name__, error=str(error))
        raise
    finally:
        report['setup_and_evaluation_wall_seconds'] = time.monotonic() - started
        save()
    return report


In [ ]:
"""Counterbalanced generation with separate grading and retained missingness."""
from datetime import datetime, timezone
import hashlib
from pathlib import Path
import time


def known_budget_ending(error, limits):
    return error in {
        f"Agent exceeded session timeout ({limits['max_time_minutes']} min)",
        f"Agent exceeded tool call budget ({limits['max_tool_calls']} calls)",
        f"Agent exceeded turns budget ({limits['max_turns']} turns)",
    }


def execute_generation_only_comparison(runner, data_dir, model_path, run_root, plan,
                                      *, checkpoint_exporter=None, session_started_monotonic=None):
    data_dir, run_root = Path(data_dir), Path(run_root)
    destination = run_root/'development_comparison.json'
    if destination.exists():
        raise FileExistsError('Preserve every previous generation')
    summarize(plan, [])
    if session_started_monotonic is None or set(plan['arm_limits']) != set(plan['arms']):
        raise ValueError('All arm limits and a pre-setup timer are required')
    if hashlib.sha256((data_dir/'tasks.jsonl').read_bytes()).hexdigest() != plan['tasks_sha256']:
        raise ValueError('Exact frozen task revision differs')
    for arm, record in plan['arms'].items():
        folder = run_root/'agents'/arm
        if any(p.is_symlink() or not p.resolve().is_relative_to(run_root.resolve()) for p in folder.rglob('*')):
            raise ValueError('Configuration path escaped')
        actual = {p.relative_to(folder).as_posix(): hashlib.sha256(p.read_bytes()).hexdigest()
                  for p in folder.rglob('*') if p.is_file()}
        if actual != record['files']:
            raise ValueError('Frozen package members differ')
    observations, runtime = [], {}
    state = dict(stopping_reason=None, grader_executed=False, independent_CPU_validation_pending=True,
                 reference_fix_or_grader_tests_passed_to_model=False, verified_repairs=None,
                 generation_success_is_not_repair_success=True)
    def save():
        result = dict(created_utc=datetime.now(timezone.utc).isoformat(), observations=observations,
                      **summarize(plan, observations), **state)
        result['unknown_grade_assignments'] = len(plan['schedule'])
        atomic_json(destination, result)
        if checkpoint_exporter:
            checkpoint_exporter(run_root)
        return result
    report = save()
    try:
        for index, assignment in enumerate(plan['schedule']):
            arm, task_id = assignment['arm'], assignment['task_id']
            limits = plan['arm_limits'][arm]
            remaining = session_started_monotonic + plan['notebook_walltime_budget_seconds'] - time.monotonic()
            required = limits['max_time_minutes']*60 + limits['timeout_seconds'] + (1200 if not runtime else 0)
            if remaining < required:
                state['stopping_reason'] = 'insufficient_allowance_before_next_assignment'
                return save()
            row = dict(assignment, result=dict(status='started', model_inference_executed=None))
            observations.append(row); save()
            print('GENERATION_START', index+1, arm, task_id, flush=True)
            try:
                row['result'] = runner(data_dir, model_path, run_root/'agents'/arm,
                    run_root/'results'/f'{index:02d}_{arm}_{task_id}', task_id, runtime=runtime, task_limits=limits)
            except Exception as error:
                row['result'] = dict(status='infrastructure_exception', error_type=type(error).__name__, error=str(error))
                state['stopping_reason'] = 'raised_infrastructure_exception'; save(); raise
            result = row['result']
            if result.get('grader_executed') is not False or result.get('official_resolved') is not None:
                state['stopping_reason'] = 'unexpected_grading_in_generation_runner'; save()
                raise ValueError('Generation must not include official grading')
            error = result.get('native_agent_error')
            if error is not None and not known_budget_ending(error, limits):
                state['stopping_reason'] = 'unrecognized_native_agent_error'
            report = save()
            print('GENERATION_RESULT', row, flush=True)
            if state['stopping_reason']:
                return report
        return report
    finally:
        server = runtime.get('server')
        if server is not None:
            try:
                server.stop()
                state['shared_server_stop_returned'] = True
            except Exception as error:
                state['shared_server_stop_returned'] = False
                state['server_stop_error_type'] = type(error).__name__
            save()


In [ ]:
if IS_KAGGLE and RUN_DECLARED_EXPERIMENT:
    result=execute_generation_only_comparison(run_generation_only_workflow_trial,DATA_DIR,MODEL_PATH,RUN,PLAN,checkpoint_exporter=export_run,session_started_monotonic=SESSION_STARTED_MONOTONIC)
    print('GENERATION_COMPARISON='+json.dumps(result))
else:
    print('Definitions only; no model, installation or grading')


In [ ]:
if IS_KAGGLE and RUN_DECLARED_EXPERIMENT:
    export_run(RUN)
